# VoxTell on Neurite-OASIS -- PLAN ROWS 17-21: image-region attribution suite (Kaggle GPU)

Only the image-region attribution rows -- rows 1-16 are assumed already run separately. Row 0 (not a
numbered plan row) is a shared baseline/GT export, run first, that every row below plots against.

| Row | Experiment | Question |
|---|---|---|
| 17 | Voxel-grid occlusion Shapley map around one organ | Which *region* of the image, if removed, changes the mask most -- the image analogue of the word-Shapley table? |
| 18 | Per-layer KL divergence between cross-attention and the GT mask | Which decoder layer's attention best matches the clinically relevant region? |
| 19 | Vanilla-gradient + Grad-CAM saliency (one patch per organ) | A cheap, approximate counterpart to row 17's exact map |
| 20 | Fused spatial feature map, matched vs. mismatched prompt | Does the model's internal image+text fusion actually move with the prompt, or stay fixed regardless of what's asked? |
| 21 | Joint Shapley over image / organ word / position word | Puts image contribution and word contribution on one common scale |

- Subjects: **1** (OASIS_OAS1_0057_MR1); organs: **cerebral cortex, cerebral white matter, cerebellum cortex, cerebellum white matter, lateral ventricle, thalamus, putamen, hippocampus, caudate**; no rotation.
- Section 3 still verifies the custom sliding-window loop against the predictor's own output before
  anything else runs -- rows 17-21 all depend on it (row 19 is the one exception: it needs gradients
  enabled, so it bypasses the loop's `inference_mode` entirely -- see its own cell).
- Output: CSVs + `row17_shapley_maps/`, `row19_saliency_maps/` under `results/`, zipped at the end.


In [ ]:
import subprocess, sys, re
from pathlib import Path

# This cell installs everything (voxtell + deps, a pinned transformers, and a clean
# torch+torchvision pair) BEFORE torch is ever imported in this kernel, then requires a
# kernel restart. Two things force this ordering:
#
# 1. `pip install voxtell` pins torch<2.9, so it can silently downgrade Kaggle's preinstalled
#    torch (observed both outcomes across runs -- sometimes torch stays put, sometimes it's
#    pulled to 2.8.x) as a side effect of installing voxtell itself.
# 2. Separately, Kaggle's preinstalled torch build has been observed shipping in a
#    self-inconsistent state regardless: Python files referencing C-level symbols (e.g.
#    torch.distributed.tensor._ops._embedding_ops._MaskPartial) that the paired compiled
#    extension doesn't actually provide -- surfaces only when importing torch's heavier
#    submodules (dynamo/FSDP), which VoxTell's predictor does via
#    `from torch._dynamo import OptimizedModule`.
#
# Both problems need a clean, paired reinstall of torch+torchvision (one pip command, no
# --no-deps, so pip picks a mutually ABI-compatible pair -- an earlier version of this fix
# reinstalled torch alone, which left the already-installed torchvision mismatched and broke
# with "RuntimeError: operator torchvision::nms does not exist"). But torch's C extension
# (`torch._C`) is a process-wide singleton: once loaded, Python can't safely swap its files
# out from under itself (tried purging sys.modules and re-importing in the same kernel --
# raises "RuntimeError: function '_has_torch_function' already has a docstring"). So this
# entire install+fix sequence has to run, and finish, WITHOUT importing torch anywhere in
# this kernel -- read the installed version via `pip show` (subprocess), never `import
# torch` -- and the kernel must be restarted afterward before anything imports torch.
_marker = Path('/kaggle/working/.env_setup_done')
if _marker.exists():
    print('environment already installed/fixed earlier in this session -- skipping. '
          f'({_marker.read_text()})')
else:
    def _pip(*args):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *args])

    print('installing voxtell + deps...')
    _pip('voxtell', 'nibabel', 'pandas', 'matplotlib', 'huggingface_hub')

    _show = subprocess.run([sys.executable, '-m', 'pip', 'show', 'torch'],
                            capture_output=True, text=True, check=True).stdout
    _torch_version = re.search(r'^Version: (.+)$', _show, re.MULTILINE).group(1).split('+')[0]
    print(f'reinstalling torch=={_torch_version} + torchvision cleanly (this can take a minute)...')
    _pip('--force-reinstall', '--no-cache-dir', f'torch=={_torch_version}', 'torchvision')

    # transformers ALONE (no other top-level packages in the same command, no --no-deps) so
    # pip resolves its own tokenizers/huggingface_hub requirements together with it -- e.g.
    # transformers 4.57.6 needs huggingface-hub<1.0,>=0.34.0 and tokenizers<=0.23.0,>=0.22.0.
    # Bundling unconstrained tokenizers/huggingface_hub upgrades into a separate command (an
    # earlier version of this fix did that) pulled newer versions that then violated
    # transformers's own pins.
    print('reinstalling transformers pinned <5...')
    _pip('--force-reinstall', '--no-cache-dir', 'transformers>=4.54.0,<5')

    _marker.write_text(f'torch=={_torch_version}')
    raise RuntimeError(
        'Environment installed/fixed. RESTART THE KERNEL NOW (Kaggle: Run -> Restart '
        'session, or the restart icon in the toolbar), then re-run this notebook from the '
        'top -- this cell will then just print "skipping" and the rest of the notebook will '
        'run against the clean, consistent install. Do not run any later cell before '
        'restarting: torch would still be broken/mismatched in THIS kernel process even '
        'though the files on disk are now fixed.'
    )


## 1. Check GPU

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "No GPU visible -- enable GPU under Notebook Settings -> Accelerator."
print("CUDA OK:", torch.cuda.get_device_name(0))


## 2. Setup Working Directories (Kaggle Environment)

In [ ]:
import os
from pathlib import Path

# /kaggle/working is the writable output directory that persists in Kaggle outputs
WORKING_DIR = Path('/kaggle/working')
RESULTS_ROOT = WORKING_DIR / 'results'
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Results will be saved under:', RESULTS_ROOT)


## 3. Install VoxTell

VoxTell is pip-installable directly (no cloning/build-from-source, no pinned-numpy fights
like Medical-SAM3 needed). PyTorch is already present in the Kaggle GPU image, so we install
VoxTell without re-pulling torch.

In [ ]:
%cd /kaggle/working

# All installation/reinstallation (voxtell + deps, transformers pin, the torch+torchvision
# consistency fix) happens in the first cell of this notebook, BEFORE torch is ever imported
# in this kernel -- see that cell for why. This cell just verifies the result.
import torch
import transformers
import voxtell
print('transformers version:', transformers.__version__)
assert int(transformers.__version__.split('.')[0]) < 5,     f"transformers {transformers.__version__} violates voxtell's own transformers<5 pin -- re-run the first cell and restart the kernel."
print('voxtell installed:', voxtell.__file__)
print('torch version:', torch.__version__, '-- CUDA available:', torch.cuda.is_available())
assert torch.cuda.is_available(), "CUDA not available -- check the first cell's pip output above."


## 4. Clone your research repo (Neurite OASIS volumes/manifest)

Uses a Kaggle Secret instead of a hardcoded token — this cell never has a real token written
into the notebook file itself.

In [ ]:
import os
import shutil
import subprocess

from kaggle_secrets import UserSecretsClient

_gh_token = UserSecretsClient().get_secret("GITHUB_PAT")
assert _gh_token, "Add a Kaggle Secret named GITHUB_PAT (Add-ons -> Secrets) before running this cell."

_clone_url = f'https://{_gh_token}@github.com/fatemafaria142/positional-bias-medical-vlms.git'
_dest = '/kaggle/working/research_repo'
shutil.rmtree(_dest, ignore_errors=True)

_result = subprocess.run(
    ['git', 'clone', '--branch', 'faria/voxtell3d-rotation-medical-segmentation', '--depth', '1', _clone_url, _dest],
    capture_output=True, text=True,
)
_stderr_scrubbed = _result.stderr.replace(_gh_token, '***')
del _gh_token, _clone_url

if _result.returncode != 0:
    raise RuntimeError(f'git clone failed (exit {_result.returncode}):\n{_stderr_scrubbed}')

RESEARCH_REPO = Path(_dest)
print('cloned into', RESEARCH_REPO)
print('repo root:', os.listdir(RESEARCH_REPO))
print('annotator folders present:', [p for p in ('faria', 'nipu', 'hridoy') if (RESEARCH_REPO / p).is_dir()])


## 5. Download and load the VoxTell model

**Correction:** the published PyPI package (`voxtell`, v0.1.0 -- confirmed by checking the
actual tagged source on GitHub, since `pip install voxtell` does not track the `main` branch)
requires `model_dir` as a **required** argument to `VoxTellPredictor` -- there is no
auto-download-on-first-use in this released version (that convenience helper exists only on
GitHub's `main` branch, not yet in a PyPI release). So we download the checkpoint ourselves
first, the same way Medical-SAM3's notebook does its checkpoint fetch, then pass the resulting
local folder in as `model_dir`.

Checkpoint: [huggingface.co/mrokuss/VoxTell](https://huggingface.co/mrokuss/VoxTell), folder
`voxtell_v1.1` (contains `plans.json` and `fold_0/checkpoint_final.pth`).

In [ ]:
import torch
from huggingface_hub import snapshot_download
from voxtell.inference.predictor import VoxTellPredictor

VOXTELL_MODEL_VERSION = 'voxtell_v1.1'
CKPT_CACHE_DIR = WORKING_DIR / 'checkpoints'
CKPT_CACHE_DIR.mkdir(parents=True, exist_ok=True)
_local_repo_dir = snapshot_download(
    repo_id='mrokuss/VoxTell',
    allow_patterns=[f'{VOXTELL_MODEL_VERSION}/*'],
    local_dir=str(CKPT_CACHE_DIR),
)
VOXTELL_MODEL_DIR = str(Path(_local_repo_dir) / VOXTELL_MODEL_VERSION)
print('VoxTell checkpoint at', VOXTELL_MODEL_DIR)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
predictor = VoxTellPredictor(model_dir=VOXTELL_MODEL_DIR, device=device)
print('VoxTell predictor ready on', device)


### 5b. Fix: text encoder OOM on a single T4/P100

VoxTell's text encoder is `Qwen/Qwen3-Embedding-4B` (4 billion parameters). The installed
v0.1.0 release loads it with `AutoModel.from_pretrained(text_encoding_model)`, no `torch_dtype`
given -- Hugging Face's default in that case is **full fp32**, so the text encoder alone is
~16GB, larger than a T4's entire 14.56GB usable memory, before the segmentation network even
factors in. This is why the smoke test OOM'd on `self.text_backbone.to(self.device)` --
it has nothing to do with passing 13 organs at once (the model dominates the memory cost, not
the number of prompts).

There's no public constructor argument to fix this, so this cell swaps in a half-precision
copy of the text encoder after the predictor is built (roughly halving its footprint to ~8GB,
which should coexist with the segmentation network), and wraps `embed_text_prompts` so its
output is always cast back to `float32` before being used further -- avoiding any fp16/fp32
dtype-mismatch risk in the segmentation network's fusion layers downstream. It reuses VoxTell's
own original `embed_text_prompts` method internally (just swaps the model and casts the
result), so none of its internal tokenization/pooling logic is duplicated here.

In [ ]:
import torch
from transformers import AutoModel, AutoTokenizer

# Guarded so re-running this cell never re-wraps an already-wrapped method --
# the true original is captured on the predictor object itself, exactly once.
if not hasattr(predictor, '_orig_embed_text_prompts'):
    predictor._orig_embed_text_prompts = predictor.embed_text_prompts

predictor.text_backbone = AutoModel.from_pretrained(
    'Qwen/Qwen3-Embedding-4B', dtype=torch.float16  # VoxTell v0.1.0's default text encoder
).eval()
# VoxTellPredictor normally lazy-loads `text_backbone` and `tokenizer` TOGETHER, guarded by a
# single `if self.text_backbone is None` check inside `_compute_text_embeddings` -- pre-setting
# `text_backbone` above (to force fp16) trips that guard early, so `tokenizer` is silently
# never loaded and stays None (-> "TypeError: 'NoneType' object is not callable" the first
# time embed_text_prompts actually runs). Load it explicitly here so both are set together.
if getattr(predictor, 'tokenizer', None) is None:
    predictor.tokenizer = AutoTokenizer.from_pretrained('Qwen/Qwen3-Embedding-4B')

@torch.inference_mode()
def _embed_text_prompts_fp16_safe(text_prompts):
    embeddings = predictor._orig_embed_text_prompts(text_prompts)
    return embeddings.float()

predictor.embed_text_prompts = _embed_text_prompts_fp16_safe
print('text encoder swapped to fp16 (output still cast back to float32); tokenizer loaded:', predictor.tokenizer is not None)


## 6. Helpers (Dice / centroid / bbox / side, same logic as scripts/common3d.py)

In [ ]:
import csv, time
from pathlib import Path

import numpy as np
import torch
from nnunetv2.imageio.nibabel_reader_writer import NibabelIOWithReorient
from acvl_utils.cropping_and_padding.bounding_boxes import insert_crop_into_image

RESEARCH_REPO = Path(RESEARCH_REPO)
reader = NibabelIOWithReorient()

_OPPOSITE = {"R": "L", "L": "R", "A": "P", "P": "A", "S": "I", "I": "S"}

def load_volume_ras(path):
    """Load a NIfTI image via VoxTell's own reader so predicted masks and ground
    truth end up in the exact same RAS-reoriented array space. Returns (1, X, Y, Z),
    the channel-first shape voxtell.inference.predictor expects."""
    data, props = reader.read_images([str(path)])
    return np.asarray(data), props

def spatial_view(arr_4d):
    a = np.asarray(arr_4d)
    while a.ndim > 3 and a.shape[0] == 1:
        a = a[0]
    return a

def load_seg_ras(path):
    data, props = reader.read_seg(str(path))
    return spatial_view(data).astype(np.int32), props

# NibabelIOWithReorient (nnU-Net's reader) returns arrays in REVERSED axis order
# relative to the canonical RAS affine -- i.e. axis0 is Superior-Inferior, axis1 is
# Anterior-Posterior, axis2 is Right-Left (not the naive (R,A,S) <-> (0,1,2) mapping).
# Verified empirically against real predictions: left/right kidney centroids differ by
# ~85 voxels on axis2 but only ~3 voxels on axis0, for every subject. Getting this wrong
# silently breaks every side/laterality computation (Dice is unaffected -- voxel overlap
# doesn't care about axis labels -- but side, centroid-shift, and causal-agreement do).
AXCODES = ("S", "A", "R")
RL_AXIS = AXCODES.index("R")

def mask_centroid_side(mask: np.ndarray):
    """Centroid + bbox + anatomical side in RAS-reoriented voxel space (see AXCODES
    above for the actual axis order). Returns (centroid, bbox, sides) where
    bbox = (min0,min1,min2,max0,max1,max2)."""
    coords = np.nonzero(mask)
    if len(coords[0]) == 0:
        return None
    centroid = tuple(float(c.mean()) for c in coords)
    bbox = tuple(int(c.min()) for c in coords) + tuple(int(c.max()) for c in coords)
    sides = tuple(code if centroid_val > mask.shape[axis] / 2 else _OPPOSITE[code]
                  for axis, (code, centroid_val) in enumerate(zip(AXCODES, centroid)))
    return centroid, bbox, sides

def dice_score(pred: np.ndarray, gt: np.ndarray) -> float:
    inter = np.logical_and(pred, gt).sum()
    denom = pred.sum() + gt.sum()
    return 1.0 if denom == 0 else float(2.0 * inter / denom)

def iou_score(a: np.ndarray, b: np.ndarray) -> float:
    inter = np.logical_and(a, b).sum()
    union = np.logical_or(a, b).sum()
    return 1.0 if union == 0 else float(inter / union)

def struct_slug(name: str) -> str:
    return name.strip().lower().replace(' ', '_').replace('/', '-')

def load_manifest_rot000(person, dataset_name):
    """Only rot000 rows -- this investigation doesn't use the rotation sweep."""
    path = RESEARCH_REPO / person / 'samples' / dataset_name / f'manifest_{dataset_name}.csv'
    with open(path, newline='') as f:
        return [row for row in csv.DictReader(f) if int(row['rotation_angle_deg']) == 0]

print('helpers ready')


import base64
import zlib


def encode_mask_rle(mask: np.ndarray) -> str:
    """Run-length-encode a boolean mask (C-order flatten), zlib-compress, base64-encode."""
    flat = np.asarray(mask, dtype=bool).ravel(order='C')
    change_idx = np.flatnonzero(flat[1:] != flat[:-1]) + 1
    boundaries = np.concatenate(([0], change_idx, [flat.size]))
    runs = np.diff(boundaries).astype(np.uint32)
    if flat.size and flat[0]:
        runs = np.concatenate(([0], runs)).astype(np.uint32)
    raw = zlib.compress(runs.tobytes(), level=9)
    return base64.b64encode(raw).decode('ascii')


def decode_mask_rle(encoded: str, shape) -> np.ndarray:
    runs = np.frombuffer(zlib.decompress(base64.b64decode(encoded)), dtype=np.uint32)
    flat = np.zeros(int(np.prod(shape)), dtype=bool)
    idx = 0
    value = False
    for run_len in runs:
        if value and run_len:
            flat[idx:idx + run_len] = True
        idx += int(run_len)
        value = not value
    return flat.reshape(shape, order='C')


## 1. Load the experiment code

In [ ]:
import json, sys, types

# The experiment code is embedded here so the notebook is self-contained. (Sources: scripts/voxtell_mechanistic.py and
# scripts/voxtell_experiments.py in the repo -- edit them there and rebuild with scripts/build_mechanistic_notebooks.py.)
_MECH_SRC = json.loads('"\\"\\"\\"\\nMechanistic helpers for the VoxTell groundedness investigation (plan rows 12-15).\\n\\nEverything here is written against the installed `voxtell` v0.1.0 source\\n(voxtell/model/voxtell_model.py, voxtell/model/transformer.py,\\nvoxtell/inference/predictor.py) and reuses the model\'s own modules -- nothing in\\nvoxtell is monkeypatched.\\n\\nFacts about VoxTell this code depends on (all verified from the source):\\n  - The image encoder never sees the text; its layer-4 features (12x12x12 tokens\\n    for a 192^3 patch) become the cross-attention *memory*.\\n  - The text embedding is projected (`project_text_embed`) and becomes the initial\\n    query state of a 6-layer pre-norm transformer decoder (`normalize_before=True`\\n    => `forward_pre`: cross-attention + FFN only, NO self-attention between the\\n    queries, so prompts in a batch never interact).\\n  - Each layer returns (state, cross-attention weights averaged over heads,\\n    shape (1, n_queries, n_tokens)).\\n  - After the last layer a LayerNorm gives the \\"mask embedding\\", which the\\n    convolutional decoder turns into a mask (one full decoder pass per prompt).\\n  - Because the decoder is a residual stream, the original text only reaches later\\n    layers through the running state. Replacing the *whole state* at layer k with\\n    another prompt\'s state therefore reproduces that prompt exactly (used as a\\n    positive control); the informative intervention is to replace a single layer\'s\\n    *update* (state_out - state_in).\\n\\nPublic API\\n  tokens_to_patch_grid, encode, base_trajectory, patched_state, decode,\\n  AttnAccumulator, run_outputs, score_masks,\\n  gradient_saliency, grad_cam_bottleneck (plan row 19),\\n  FusedSpatialCapture (plan row 20)\\n\\"\\"\\"\\n\\nimport contextlib\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\nfrom einops import rearrange\\n\\n\\n# ---------------------------------------------------------------------------------\\n# token <-> voxel bookkeeping\\n# ---------------------------------------------------------------------------------\\ndef tokens_to_patch_grid(w, g):\\n    \\"\\"\\"`w`: (..., g**3) values over VoxTell\'s image tokens -> (..., g, g, g) indexed in\\n    the patch\'s own axis order (axis0, axis1, axis2).\\n\\n    VoxTell flattens tokens as `rearrange(feat, \'b c d h w -> (h w d) b c\')`, i.e.\\n    token index = (h*g + w)*g + d, where (d, h, w) are the patch axes (0, 1, 2).\\n    \\"\\"\\"\\n    lead = w.shape[:-1]\\n    x = w.reshape(*lead, g, g, g)  # [h, w, d] == [axis1, axis2, axis0]\\n    nd = x.ndim\\n    return x.permute(*range(nd - 3), nd - 1, nd - 3, nd - 2)  # -> [d, h, w]\\n\\n\\n# ---------------------------------------------------------------------------------\\n# model pieces (mirror VoxTellModel.forward, split so we can read / patch the decoder)\\n# ---------------------------------------------------------------------------------\\ndef encode(model, patch):\\n    \\"\\"\\"patch: (1, C, X, Y, Z) -> (skips, memory (S, 1, Q)).\\"\\"\\"\\n    skips = model.encoder(patch)\\n    feat = skips[model.selected_decoder_layer]\\n    mem = rearrange(feat, \'b c d h w -> b h w d c\')\\n    mem = model.project_bottleneck_embed(mem)\\n    mem = rearrange(mem, \'b h w d c -> (h w d) b c\')\\n    return skips, mem\\n\\n\\ndef base_trajectory(model, tgt0, memory, pos):\\n    \\"\\"\\"Run the decoder layers on `tgt0` (M, 1, Q). Returns (states, attns):\\n    states[k] is the state entering layer k (states[-1] = after the last layer, before\\n    the final norm); attns[k] = (1, M, S) cross-attention weights of layer k.\\"\\"\\"\\n    layers = model.transformer_decoder.layers\\n    assert all(l.normalize_before for l in layers), \'expected the pre-norm (forward_pre) decoder\'\\n    states, attns = [tgt0], []\\n    t = tgt0\\n    for layer in layers:\\n        t, w = layer(t, memory, pos=pos)\\n        states.append(t)\\n        attns.append(w)\\n    return states, attns\\n\\n\\ndef patched_state(model, tgt0_row, memory, pos, base_states, ops):\\n    \\"\\"\\"Re-run the decoder for ONE query (tgt0_row: (1, 1, Q)) with interventions.\\n\\n    ops: list of dicts {\'layer\': k, \'mode\': \'update\' | \'state\', \'src\': row}\\n      - \'update\': the layer-k output is `state_in + (src\'s layer-k update)`, i.e. only that\\n                  layer\'s contribution is taken from the source prompt\\n      - \'state\' : the layer-k output is the source prompt\'s whole state (positive control)\\n    `base_states` is the list returned by `base_trajectory` for all prompts (M rows).\\"\\"\\"\\n    by_layer = {}\\n    for op in ops:\\n        by_layer.setdefault(op[\'layer\'], []).append(op)\\n    t = tgt0_row\\n    for k, layer in enumerate(model.transformer_decoder.layers):\\n        new, _ = layer(t, memory, pos=pos)\\n        for op in by_layer.get(k, []):\\n            s = op[\'src\']\\n            if op[\'mode\'] == \'update\':\\n                new = t + (base_states[k + 1][s:s + 1] - base_states[k][s:s + 1])\\n            elif op[\'mode\'] == \'state\':\\n                new = base_states[k + 1][s:s + 1].clone()\\n            else:\\n                raise ValueError(op[\'mode\'])\\n        t = new\\n    return t\\n\\n\\ndef decode(model, skips, mask_emb):\\n    \\"\\"\\"mask_emb: (n, Q) -> logits (n, X, Y, Z). Same per-prompt loop as VoxTellModel.forward.\\"\\"\\"\\n    me = mask_emb.unsqueeze(0)  # (1, n, Q)\\n    mes = [proj(me) for proj in model.project_to_decoder_channels]\\n    outs = []\\n    for i in range(me.shape[1]):\\n        prompt_embeds = [m[:, i:i + 1] for m in mes]\\n        outs.append(model.decoder(skips, prompt_embeds)[0])\\n    return torch.cat(outs, dim=1)[0]\\n\\n\\n# ---------------------------------------------------------------------------------\\n# attention accumulation over the sliding window\\n# ---------------------------------------------------------------------------------\\nclass AttnAccumulator:\\n    \\"\\"\\"Accumulates cross-attention *density* (attention weight x n_tokens, so the mean is 1)\\n    into voxel space, weighted by the same Gaussian as the logits.\\"\\"\\"\\n\\n    def __init__(self, spatial_shape, n_layers, n_rows, patch_size, device):\\n        self.buf = torch.zeros((n_layers, n_rows, *spatial_shape), dtype=torch.float32, device=device)\\n        self.cnt = torch.zeros(tuple(spatial_shape), dtype=torch.float32, device=device)\\n        self.patch_size = tuple(int(p) for p in patch_size)\\n\\n    def add(self, slicer, attn, gaussian):\\n        \\"\\"\\"attn: (L, M, S) float; gaussian: patch-shaped weights.\\"\\"\\"\\n        L, M, S = attn.shape\\n        g = int(round(S ** (1.0 / 3.0)))\\n        assert g ** 3 == S, f\'{S} tokens is not a cube\'\\n        dens = tokens_to_patch_grid(attn.float(), g) * S  # (L, M, g, g, g)\\n        gz = gaussian.float()\\n        for l in range(L):\\n            up = F.interpolate(dens[l][None], size=self.patch_size, mode=\'nearest\')[0]  # (M, X, Y, Z)\\n            self.buf[(l, slice(None), *slicer[1:])] += up * gz\\n        self.cnt[slicer[1:]] += gz\\n\\n    def finalize(self, revert_slicer):\\n        out = self.buf / self.cnt.clamp_min(1e-6)\\n        return out[(slice(None), slice(None), *revert_slicer[1:])]\\n\\n\\n# ---------------------------------------------------------------------------------\\n# sliding-window runner (same padding / step / Gaussian / accumulation as the predictor)\\n# ---------------------------------------------------------------------------------\\n@torch.inference_mode()\\ndef run_outputs(predictor, data, embeddings, outputs, attn_acc_factory=None, attn_rows=None, on_patch=None):\\n    \\"\\"\\"Sliding-window inference for several *outputs* that share one encoder pass per patch.\\n\\n    data        : preprocessed image tensor (C, X, Y, Z) (from predictor.preprocess)\\n    embeddings  : (1, M, D) text embeddings (rows may be blends built by the caller)\\n    outputs     : list of {\'name\': str, \'row\': int, \'ops\': [...]}; `ops` as in `patched_state`\\n    attn_acc_factory : optional callable(spatial_shape, n_layers, n_rows) -> AttnAccumulator\\n    attn_rows        : accumulate attention only for the first `attn_rows` embedding rows (default: all rows)\\n    on_patch         : optional callable(slicer, states, attns, ctx) called for every patch; `states` = list of\\n                       7 tensors (M, 1, Q) (input state + output of each layer), `attns` = 6 tensors (1, M, S),\\n                       patch_ctx = {\'revert\': padding-revert slicer, \'padded_shape\': spatial shape after padding}\\n    Returns (logits (n_out, X, Y, Z) half, attention (L, M, X, Y, Z) or None), padding reverted.\\"\\"\\"\\n    from acvl_utils.cropping_and_padding.padding import pad_nd_image\\n    from nnunetv2.inference.sliding_window_prediction import compute_gaussian\\n\\n    device = predictor.device\\n    model = predictor.network.to(device)\\n    data_p, revert = pad_nd_image(data, predictor.patch_size, \'constant\', {\'value\': 0}, True, None)\\n    slicers = predictor._internal_get_sliding_window_slicers(data_p.shape[1:])\\n    data_p = data_p.to(device)\\n    emb = embeddings[0].to(device)  # (M, D)\\n\\n    n_out = len(outputs)\\n    logits = torch.zeros((n_out, *data_p.shape[1:]), dtype=torch.half, device=device)\\n    npred = torch.zeros(data_p.shape[1:], dtype=torch.half, device=device)\\n    gaussian = compute_gaussian(tuple(predictor.patch_size), sigma_scale=1. / 8,\\n                                value_scaling_factor=10, device=device)\\n    attn_acc = None\\n    if attn_acc_factory is not None:\\n        n_attn = attn_rows if attn_rows is not None else emb.shape[0]\\n        attn_acc = attn_acc_factory(tuple(data_p.shape[1:]), len(model.transformer_decoder.layers), n_attn)\\n    patch_ctx = {\\"revert\\": revert, \\"padded_shape\\": tuple(data_p.shape[1:])}\\n    ctx = torch.autocast(device.type) if device.type == \'cuda\' else contextlib.nullcontext()\\n\\n    for slicer in slicers:\\n        patch = data_p[slicer][None]\\n        with ctx:\\n            skips, memory = encode(model, patch)\\n            tgt0 = model.project_text_embed(emb).unsqueeze(1)  # (M, 1, Q)\\n            states, attns = base_trajectory(model, tgt0, memory, model.pos_embed)\\n            embs = []\\n            for o in outputs:\\n                r = o[\'row\']\\n                if o.get(\'ops\'):\\n                    t = patched_state(model, tgt0[r:r + 1], memory, model.pos_embed, states, o[\'ops\'])\\n                else:\\n                    t = states[-1][r:r + 1]\\n                embs.append(model.transformer_decoder.norm(t))\\n            mask_emb = torch.cat(embs, dim=0)[:, 0]  # (n_out, Q)\\n            pred = decode(model, skips, mask_emb)\\n        pred = (pred.float() * gaussian.float()).half()\\n        logits[(slice(None), *slicer[1:])] += pred\\n        npred[slicer[1:]] += gaussian.half()\\n        if attn_acc is not None:\\n            stack = torch.stack([a[0] for a in attns]).float()   # (L, M, S)\\n            attn_acc.add(slicer, stack[:, :n_attn], gaussian)\\n        if on_patch is not None:\\n            on_patch(slicer, states, attns, patch_ctx)\\n\\n    torch.div(logits, npred, out=logits)\\n    if torch.any(torch.isinf(logits)):\\n        raise RuntimeError(\'inf in logits\')\\n    logits = logits[(slice(None), *revert[1:])]\\n    attn = attn_acc.finalize(revert) if attn_acc is not None else None\\n    return logits, attn\\n\\n\\n# ---------------------------------------------------------------------------------\\n# scoring (numpy, side-agnostic: uses overlap with the two candidate ground-truth organs)\\n# ---------------------------------------------------------------------------------\\ndef _dice(a, b):\\n    d = a.sum() + b.sum()\\n    return 1.0 if d == 0 else float(2.0 * np.logical_and(a, b).sum() / d)\\n\\n\\ndef score_masks(mask, gt_a, gt_b, cov_thr=0.3, other_thr=0.2):\\n    \\"\\"\\"Overlap of a predicted mask with two candidate ground-truth organs.\\n    cov_x = |mask & gt_x| / |gt_x|. `call` is \'a\' / \'b\' when the mask covers that organ\\n    (>= cov_thr) and the other one less than other_thr, \'both\' when both are covered,\\n    \'empty\' for an empty mask and \'neither\' otherwise.\\"\\"\\"\\n    mask, gt_a, gt_b = (np.asarray(x, dtype=bool) for x in (mask, gt_a, gt_b))\\n    n = int(mask.sum())\\n    cov_a = float(np.logical_and(mask, gt_a).sum() / max(gt_a.sum(), 1))\\n    cov_b = float(np.logical_and(mask, gt_b).sum() / max(gt_b.sum(), 1))\\n    if n == 0:\\n        call = \'empty\'\\n    elif cov_a >= cov_thr and cov_b >= cov_thr:\\n        call = \'both\'\\n    elif cov_a >= cov_thr and cov_b < other_thr:\\n        call = \'a\'\\n    elif cov_b >= cov_thr and cov_a < other_thr:\\n        call = \'b\'\\n    else:\\n        call = \'neither\'\\n    return {\'n_pred_voxels\': n, \'cov_a\': round(cov_a, 4), \'cov_b\': round(cov_b, 4),\\n            \'dice_a\': round(_dice(mask, gt_a), 4), \'dice_b\': round(_dice(mask, gt_b), 4), \'call\': call}\\n\\n\\n# ---------------------------------------------------------------------------------\\n# plan row 19 -- gradient-based saliency (vanilla gradient + Grad-CAM on the bottleneck)\\n# ---------------------------------------------------------------------------------\\ndef gradient_saliency(model, patch, row=0, memory=None, pos=None, prompts_embed=None):\\n    \\"\\"\\"One-patch vanilla-gradient saliency: d(sum of logits)/d(patch), for the prompt at `row`.\\n\\n    MUST be called OUTSIDE any `torch.inference_mode()`/`torch.no_grad()` context -- tensors\\n    created under `inference_mode` are permanently non-differentiable even once you leave it\\n    (this is why `run_outputs` is `@torch.inference_mode()` and cannot be reused here; call\\n    `encode`/`decode` directly instead, as this function does). The taint also applies to the\\n    MODEL\'S OWN WEIGHTS, not just inputs: if `model`\'s parameters were ever created/touched\\n    while some earlier code ran under `inference_mode` (e.g. inside predictor construction, a\\n    detail of `voxtell`\'s own loading path, not this file), `model.parameters()` can themselves\\n    be inference tensors -- `conv3d` then raises \\"Inference tensors cannot be saved for backward\\"\\n    on the very first encoder layer, regardless of whether the input patch is clean, because\\n    autograd needs the weight (not just the input) to compute the input\'s own gradient. Cloning\\n    each tainted parameter once, in place, outside any inference_mode context (below), permanently\\n    de-taints the model for every future call -- confirmed locally against a real `nn.Conv3d`\\n    with weights deliberately tainted the same way (forward+backward fails before this fix,\\n    succeeds after it, with the clone changing nothing numerically).\\n\\n    patch: (1, C, X, Y, Z), will be cloned and marked requires_grad. prompts_embed: (M, D) text\\n    embeddings for every prompt sharing this patch (only `row`\'s logits are backpropagated, but\\n    all prompts run through the same encoder pass, matching `run_outputs`\'s batching).\\n\\n    Returns (saliency (X, Y, Z) float32 = |d logits / d patch| summed over channels,\\n             bottleneck_grad (g, g, g) float32 = Grad-CAM weights at the bottleneck feature map,\\n             i.e. the channel-mean gradient of the same logits w.r.t. the bottleneck token grid).\\n    \\"\\"\\"\\n    assert torch.is_grad_enabled(), \'gradient_saliency needs grad enabled -- do not call inside inference_mode/no_grad\'\\n    for p in model.parameters():  # one-time de-taint; cheap no-op on every call after the first\\n        if p.is_inference():\\n            p.data = p.data.clone()\\n    patch = patch.clone().detach().requires_grad_(True)\\n    skips, mem = encode(model, patch)\\n    # keep the bottleneck activation\'s own grad (for Grad-CAM) by not detaching it\\n    bottleneck = skips[model.selected_decoder_layer]\\n    bottleneck.retain_grad()\\n    tgt0 = model.project_text_embed(prompts_embed).unsqueeze(1)  # (M, 1, Q)\\n    states, _ = base_trajectory(model, tgt0, mem, pos if pos is not None else model.pos_embed)\\n    mask_emb = model.transformer_decoder.norm(states[-1][row:row + 1])  # (1, Q)\\n    logits = decode(model, skips, mask_emb)  # (1, X, Y, Z)\\n    logits.sum().backward()\\n    saliency = patch.grad.detach().abs().sum(dim=1)[0].float()  # (X, Y, Z)\\n    cam = bottleneck.grad.detach().mean(dim=1)[0].float()  # (d, h, w) -- channel-mean, ReLU\'d below\\n    cam = cam.clamp_min(0)\\n    return saliency.cpu().numpy(), cam.cpu().numpy()\\n\\n\\ndef grad_cam_upsample(cam_grid, patch_size):\\n    \\"\\"\\"Upsample a (d, h, w) Grad-CAM grid (as returned by gradient_saliency\'s bottleneck_grad,\\n    already in the patch\'s own axis order via tokens_to_patch_grid\'s convention) to voxel space.\\"\\"\\"\\n    g = cam_grid.shape[0]\\n    t = torch.from_numpy(cam_grid)[None, None].float()\\n    up = F.interpolate(t, size=tuple(patch_size), mode=\'trilinear\', align_corners=False)\\n    return up[0, 0].numpy()\\n\\n\\n# ---------------------------------------------------------------------------------\\n# plan row 20 -- fused spatial feature map (resolves rows 1/4\'s \\"needs hook\\")\\n# ---------------------------------------------------------------------------------\\nclass FusedSpatialCapture:\\n    \\"\\"\\"Captures VoxTellDecoder\'s per-stage spatial `fusion_features` (the image+text product that\\n    actually varies with the prompt at full spatial resolution, as opposed to the single global\\n    `mask_emb` query vector used elsewhere in this file, OR the final pre-sigmoid logits that plan\\n    row 1 already used as its stand-in \\"internal representation\\" -- this is a genuine intermediate\\n    layer, not the output) via a forward_pre_hook on each of the decoder\'s `seg_layers` stages --\\n    `fusion_features` is a local variable inside `VoxTellDecoder.forward` (voxtell_model.py), not a\\n    module\'s own output, so it is read off of the *input* to the next stage that consumes it, not a\\n    forward_hook\'s output.\\n\\n    Usage:\\n        cap = FusedSpatialCapture(model.decoder)\\n        cap.attach()\\n        ... run inference (predictor.predict_* or vm.decode) ...\\n        maps = cap.maps  # {stage_index: (B, 1, H, W, D) tensor}, resolutions 24/48/96/192**3\\n        cap.detach()\\n\\n    UNVERIFIED: the exact submodule path to `model.decoder.seg_layers` (or equivalent) has not\\n    been confirmed against a live model/forward pass -- the first real run should print\\n    `{k: v.shape for k, v in maps.items()}` and sanity-check it against the four documented\\n    resolutions (24, 48, 96, 192 cubed) before trusting any IoU computed from these maps.\\n    \\"\\"\\"\\n\\n    def __init__(self, decoder_module, stage_attr=\'seg_layers\'):\\n        self.decoder_module = decoder_module\\n        self.stage_attr = stage_attr\\n        self.maps = {}\\n        self._handles = []\\n\\n    def _make_hook(self, stage_idx):\\n        def hook(module, inputs):\\n            self.maps[stage_idx] = inputs[0].detach().cpu()\\n        return hook\\n\\n    def attach(self):\\n        stages = getattr(self.decoder_module, self.stage_attr)\\n        for i, stage in enumerate(stages):\\n            self._handles.append(stage.register_forward_pre_hook(self._make_hook(i)))\\n        return self\\n\\n    def detach(self):\\n        for h in self._handles:\\n            h.remove()\\n        self._handles = []\\n\\n\\n\\n# ---------------------------------------------------------------------------------\\n# plan row 24 -- cross-attention head ablation (which specific head, not just which layer)\\n# ---------------------------------------------------------------------------------\\nclass HeadAblation:\\n    \\"\\"\\"Zero-ablates one or more cross-attention HEADS (not a whole layer) at one or more decoder\\n    layers. This is the standard \\"zero ablation\\" used in mechanistic interpretability to test\\n    whether one specific head is causally responsible for a behavior, as opposed to row 15/18\'s\\n    whole-layer attention, which cannot distinguish \\"one head does this\\" from \\"every head does\\n    this a little\\".\\n\\n    IMPLEMENTATION NOTE (verified locally against a real `nn.MultiheadAttention`, see the module\\n    docstring\'s test below -- do not \\"simplify\\" this back to a `forward_pre_hook` on `out_proj`:\\n    that was the first attempt here and it silently never fires, because PyTorch\'s internal\\n    `multi_head_attention_forward` calls `F.linear(attn_output, out_proj.weight, out_proj.bias)`\\n    directly, NOT `out_proj(...)` -- so a hook on the `out_proj` module itself is never invoked).\\n\\n    Instead, this hooks `multihead_attn` itself with a `forward_hook` (`with_kwargs=True`) that:\\n      1. Re-invokes the SAME call (same args/kwargs) with `need_weights=True,\\n         average_attn_weights=False` to obtain the REAL per-head attention weights (a reentrancy\\n         guard prevents this inner call from re-triggering the hook).\\n      2. Recomputes only the value projection (`in_proj_weight`/`in_proj_bias`\'s documented\\n         `[2*embed_dim:3*embed_dim]` V slice -- the one deliberately \\"manual\\" piece here, and the\\n         lowest-risk one: a single Linear split, not a softmax/scaling reimplementation) and\\n         combines it with those REAL attention weights to get each head\'s own output contribution.\\n      3. Zeroes the targeted head(s)\' contribution, re-applies `out_proj`, and returns\\n         `(ablated_output, <the ORIGINAL second return value>)` -- so any code reading the second\\n         element (attention weights, e.g. `AttnAccumulator`) sees the same shape/semantics as an\\n         unhooked call; only the first element (the actual output used by the residual stream) is\\n         changed.\\n    With the head list empty, this reconstruction is mathematically exact (confirmed to match the\\n    unhooked output bit-for-bit locally); `validate_head_ablation_noop` (below) re-confirms this on\\n    the real model before trusting any row 24 result.\\n\\n    Usage:\\n        ablation = HeadAblation(model.transformer_decoder.layers, layer_indices=[5], head_indices=[2])\\n        ablation.attach()\\n        ... run inference (predictor.predict_* or vm.decode) ...\\n        ablation.detach()\\n\\n    UNVERIFIED (not the hook mechanism above, which is verified -- this is the one remaining\\n    assumption): `layer.multihead_attn` is the cross-attention `nn.MultiheadAttention` submodule,\\n    with a combined `in_proj_weight` (i.e. `_qkv_same_embed_dim=True` -- query/key/value share one\\n    embedding dimension, the common case and the only one this class supports; `attach()` raises\\n    `AttributeError` immediately, not a silent wrong answer, if `in_proj_weight` is None). DETR-style\\n    decoder layer naming, matching this file\'s own `forward_pre`/pre-norm comments at the top of\\n    `base_trajectory`.\\n    \\"\\"\\"\\n\\n    def __init__(self, decoder_layers, layer_indices, head_indices):\\n        self.decoder_layers = decoder_layers\\n        self.layer_indices = list(layer_indices)\\n        self.head_indices = list(head_indices)\\n        self._handles = []\\n\\n    def _make_hook(self, mha):\\n        heads = list(self.head_indices)\\n        embed_dim, num_heads = mha.embed_dim, mha.num_heads\\n        head_dim = embed_dim // num_heads\\n        guard = {\\"active\\": False}\\n\\n        def hook(module, args, kwargs, output):\\n            if guard[\\"active\\"]:\\n                return output  # the inner re-invocation below; let it through unmodified\\n            guard[\\"active\\"] = True\\n            try:\\n                kw2 = dict(kwargs)\\n                kw2[\\"need_weights\\"] = True\\n                kw2[\\"average_attn_weights\\"] = False\\n                with torch.no_grad():\\n                    _, per_head_w = module(*args, **kw2)  # (bsz, num_heads, tgt_len, src_len)\\n            finally:\\n                guard[\\"active\\"] = False\\n            if not heads:\\n                return output  # nothing to ablate -- skip the reconstruction entirely\\n            value = kwargs.get(\\"value\\", args[2] if len(args) > 2 else None)\\n            assert value is not None, \\"HeadAblation: could not find the `value` argument in this call\\"\\n            assert mha.in_proj_weight is not None, \\\\\\n                \\"HeadAblation only supports combined in_proj_weight (query/key/value sharing one embed_dim)\\"\\n            wv = mha.in_proj_weight[2 * embed_dim:3 * embed_dim]\\n            bv = mha.in_proj_bias[2 * embed_dim:3 * embed_dim] if mha.in_proj_bias is not None else None\\n            v = F.linear(value, wv, bv)  # (src_len, bsz, embed_dim)\\n            src_len, bsz, _ = v.shape\\n            tgt_len = per_head_w.shape[-2]\\n            v = v.reshape(src_len, bsz, num_heads, head_dim).permute(1, 2, 0, 3)  # (bsz, heads, src_len, head_dim)\\n            head_out = (per_head_w @ v).clone()  # (bsz, heads, tgt_len, head_dim)\\n            head_out[:, heads, :, :] = 0\\n            concat = head_out.permute(2, 0, 1, 3).reshape(tgt_len, bsz, embed_dim)\\n            ablated = F.linear(concat, mha.out_proj.weight, mha.out_proj.bias)\\n            return (ablated, output[1])\\n\\n        return hook\\n\\n    def attach(self):\\n        for k in self.layer_indices:\\n            mha = self.decoder_layers[k].multihead_attn\\n            self._handles.append(mha.register_forward_hook(self._make_hook(mha), with_kwargs=True))\\n        return self\\n\\n    def detach(self):\\n        for h in self._handles:\\n            h.remove()\\n        self._handles = []\\n\\n\\ndef num_cross_attn_heads(decoder_layers, layer_idx=0):\\n    \\"\\"\\"`multihead_attn.num_heads` of one decoder layer -- read at runtime, never hardcoded.\\"\\"\\"\\n    return decoder_layers[layer_idx].multihead_attn.num_heads\\n\\n\\ndef validate_head_ablation_noop(predict_fn, layer_idx, decoder_layers):\\n    \\"\\"\\"Attaches a HeadAblation with an EMPTY head list at a real layer (so the hook is live, but\\n    zeroes nothing) and checks the prediction is unchanged. `predict_fn` is a zero-arg callable\\n    that runs one prediction and returns a boolean mask (e.g. `lambda: predict_masks(predictor,\\n    img_arr, [prompt])[0]`). Returns the fraction of voxels that agree between the hooked and\\n    unhooked predictions -- must be 1.0 (or, allowing for floating-point rounding, very close)\\n    before any row 24 result can be trusted.\\"\\"\\"\\n    base = predict_fn()\\n    ablation = HeadAblation(decoder_layers, [layer_idx], [])\\n    ablation.attach()\\n    try:\\n        noop = predict_fn()\\n    finally:\\n        ablation.detach()\\n    import numpy as np\\n    return float((np.asarray(base) == np.asarray(noop)).mean())\\n\\n\\ndef activation_overlap_iou(fused_map, gt_bbox_mask, threshold_quantile=0.9):\\n    \\"\\"\\"Plan row 1/20\'s metric: IoU between the top-`1 - threshold_quantile` activations of a\\n    (possibly multi-channel) fused spatial map, upsampled to `gt_bbox_mask`\'s shape, and the\\n    ground-truth region. `fused_map`: (C, H, W, D) or (H, W, D) numpy/tensor; `gt_bbox_mask`:\\n    (X, Y, Z) bool.\\"\\"\\"\\n    m = torch.as_tensor(fused_map).float()\\n    if m.ndim == 4:\\n        m = m.abs().mean(dim=0)\\n    m = m[None, None]\\n    up = F.interpolate(m, size=tuple(gt_bbox_mask.shape), mode=\'trilinear\', align_corners=False)[0, 0]\\n    thr = torch.quantile(up.flatten(), threshold_quantile)\\n    hot = (up >= thr).numpy()\\n    gt = np.asarray(gt_bbox_mask, dtype=bool)\\n    inter = np.logical_and(hot, gt).sum()\\n    union = np.logical_or(hot, gt).sum()\\n    return float(inter / union) if union else 0.0\\n"')
_EXP_SRC = json.loads('"\\"\\"\\"\\nExperiments for plan rows 12-16 and 17-21 (all take a VoxTellPredictor-like object and plain numpy\\narrays, so they can be tested without a GPU).\\n\\n  row 12  run_flip                image left-right flip, prompt unchanged\\n  row 13  run_erasure             organ filled with noise / both organs filled with noise / noise image\\n  row 14  run_interventions       embedding blend and per-layer patching between \\"left X\\" and \\"right X\\"\\n  row 15  (same run)              cross-attention of every prompt over every organ region, per decoder layer\\n  row 16  (same run)              distance between the left and right query states at every decoder layer\\n  row 17  run_grid_occlusion      voxel-grid occlusion Shapley/SHAP map around one organ\\n  row 18  run_attention_kl        per-layer KL divergence between cross-attention and the GT organ mask\\n  row 19  run_gradient_saliency   vanilla-gradient + Grad-CAM saliency, one patch around the organ\\n  row 20  run_activation_overlap  fused spatial feature map overlap, matched vs. mismatched prompt\\n  row 21  run_joint_image_word_shapley   image (intact/noised) x organ-word x position-word, 3-role Shapley\\n  row 22  run_laterality_probing  linear probe per decoder layer: is left/right linearly decodable, not just different?\\n  row 23  run_embedding_geometry  is there one consistent \\"left -> right\\" direction across organs, or a different one per organ?\\n  row 24  run_attention_head_ablation   zero-ablate one cross-attention head at a time -- which head, not just which layer\\n\\nArrays: `img_arr` is (1, X, Y, Z) in the RAS-reoriented grid used by the notebooks, `label_vol` is (X, Y, Z).\\n`rl_axis` is the right-left axis of the spatial arrays (2 in the notebooks).\\n\\"\\"\\"\\n\\nimport base64\\nimport itertools\\nimport zlib\\n\\nimport numpy as np\\nimport torch\\n\\nimport voxtell_mechanistic as vm\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# RLE mask encode/decode (duplicated, not imported, from build_groundedness_investigation_notebooks.py\\n# -- same self-contained-module convention as the Shapley helpers above). Used so every row\'s CSV can\\n# carry the full predicted/GT mask, not just summary numbers, for later plotting (image + GT + mask\\n# overlays) exactly like the rotation-sweep CSVs already do.\\n# ------------------------------------------------------------------------------------------------\\ndef encode_mask_rle(mask):\\n    \\"\\"\\"Run-length-encode a boolean mask (C-order flatten), zlib-compress, base64-encode.\\"\\"\\"\\n    flat = np.asarray(mask, dtype=bool).ravel(order=\'C\')\\n    change_idx = np.flatnonzero(flat[1:] != flat[:-1]) + 1\\n    boundaries = np.concatenate(([0], change_idx, [flat.size]))\\n    runs = np.diff(boundaries).astype(np.uint32)\\n    if flat.size and flat[0]:\\n        runs = np.concatenate(([0], runs)).astype(np.uint32)\\n    raw = zlib.compress(runs.tobytes(), level=9)\\n    return base64.b64encode(raw).decode(\'ascii\')\\n\\n\\ndef decode_mask_rle(encoded, shape):\\n    runs = np.frombuffer(zlib.decompress(base64.b64decode(encoded)), dtype=np.uint32)\\n    flat = np.zeros(int(np.prod(shape)), dtype=bool)\\n    pos, val = 0, False\\n    for r in runs:\\n        if val:\\n            flat[pos:pos + r] = True\\n        pos += r\\n        val = not val\\n    return flat.reshape(shape)\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# shared Shapley machinery (duplicated, not imported, from build_groundedness_investigation_notebooks.py\'s\\n# rows 5-7 -- kept self-contained here since this module, like voxtell_mechanistic.py, is embedded\\n# verbatim into the mechanistic notebooks; see that file\'s `shapley_values`/`all_role_combos` for the\\n# word-attribution use of the exact same math)\\n# ------------------------------------------------------------------------------------------------\\ndef all_role_combos(roles):\\n    \\"\\"\\"All 2^n present/absent combinations. Returns list of (frozenset(present roles), present_dict).\\"\\"\\"\\n    combos = []\\n    for bits in itertools.product([True, False], repeat=len(roles)):\\n        present = dict(zip(roles, bits))\\n        key = frozenset(r for r in roles if present[r])\\n        combos.append((key, present))\\n    return combos\\n\\n\\ndef shapley_values(v, roles):\\n    \\"\\"\\"Exact Shapley values. v: dict[frozenset(present roles)] -> payoff (float). roles: full player set N.\\"\\"\\"\\n    import math\\n    n = len(roles)\\n    phi = {r: 0.0 for r in roles}\\n    for i in roles:\\n        others = [r for r in roles if r != i]\\n        for bits in itertools.product([True, False], repeat=len(others)):\\n            s = frozenset(r for r, b in zip(others, bits) if b)\\n            weight = math.factorial(len(s)) * math.factorial(n - len(s) - 1) / math.factorial(n)\\n            phi[i] += weight * (v[s | {i}] - v[s])\\n    return phi\\n\\n\\n# ------------------------------------------------------------------------------------------------\\ndef mirror(organ):\\n    \\"\\"\\"\'left kidney\' <-> \'right kidney\'.\\"\\"\\"\\n    side, core = organ.split(None, 1)\\n    return (\\"right \\" if side == \\"left\\" else \\"left \\") + core\\n\\n\\ndef centroid(mask):\\n    idx = np.nonzero(mask)\\n    return np.array([c.mean() for c in idx]) if len(idx[0]) else None\\n\\n\\ndef _to_full_masks(logits, bbox, orig_shape):\\n    from acvl_utils.cropping_and_padding.bounding_boxes import insert_crop_into_image\\n    with torch.no_grad():\\n        cropped = torch.cat([(torch.sigmoid(logits[i:i + 8].float()) > 0.5).to(\\"cpu\\", torch.uint8)\\n                             for i in range(0, logits.shape[0], 8)]).numpy()\\n    full = np.zeros((cropped.shape[0], *orig_shape), dtype=np.uint8)\\n    return insert_crop_into_image(full, cropped, bbox).astype(bool)\\n\\n\\ndef predict_masks(predictor, img_arr, prompts, chunk=32):\\n    \\"\\"\\"Same steps as the notebooks: preprocess, embed, sliding window, threshold, put back into the full volume.\\"\\"\\"\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    emb = predictor.embed_text_prompts(list(prompts))\\n    parts = []\\n    for s in range(0, emb.shape[1], chunk):\\n        parts.append(predictor.predict_sliding_window_return_logits(data, emb[:, s:s + chunk]).to(\\"cpu\\"))\\n        torch.cuda.empty_cache() if torch.cuda.is_available() else None\\n    return _to_full_masks(torch.cat(parts, dim=0), bbox, orig_shape)\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 12 -- image left-right flip\\n# ------------------------------------------------------------------------------------------------\\ndef run_flip(predictor, img_arr, label_vol, organs, ids, prompts, rl_axis=2):\\n    \\"\\"\\"Flip the image and its labels along the right-left axis and keep every prompt unchanged.\\n    call \'a\' = the mask covers the organ that the prompt names (follows the anatomy / the label);\\n    call \'b\' = the mask covers the organ on the other side (follows the side of the image).\\"\\"\\"\\n    flipped_img = np.ascontiguousarray(np.flip(img_arr, axis=rl_axis + 1))\\n    flipped_lab = np.ascontiguousarray(np.flip(label_vol, axis=rl_axis))\\n    rows = []\\n    for cond, img, lab in ((\\"original\\", img_arr, label_vol), (\\"flipped\\", flipped_img, flipped_lab)):\\n        masks = predict_masks(predictor, img, [prompts[o] for o in organs])\\n        for k, o in enumerate(organs):\\n            sc = vm.score_masks(masks[k], lab == ids[o], lab == ids[mirror(o)])\\n            rows.append({\\"organ\\": o, \\"condition\\": cond, \\"prompt\\": prompts[o], \\"n_pred_voxels\\": sc[\\"n_pred_voxels\\"],\\n                         \\"cov_named_organ\\": sc[\\"cov_a\\"], \\"cov_other_side_organ\\": sc[\\"cov_b\\"],\\n                         \\"dice_named_organ\\": sc[\\"dice_a\\"], \\"dice_other_side_organ\\": sc[\\"dice_b\\"],\\n                         \\"call\\": {\\"a\\": \\"follows_named_organ\\", \\"b\\": \\"follows_image_side\\"}.get(sc[\\"call\\"], sc[\\"call\\"])})\\n    return rows\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 13 -- organ replaced by noise / noise image\\n# ------------------------------------------------------------------------------------------------\\ndef noise_region(img_arr, region, rng):\\n    \\"\\"\\"Replace the voxels of `region` by Gaussian noise with the mean and spread of the image\'s non-zero voxels.\\"\\"\\"\\n    out = img_arr.copy()\\n    vals = img_arr[img_arr != 0]\\n    out[0][region] = rng.normal(float(vals.mean()), float(vals.std()), size=int(region.sum()))\\n    return out\\n\\n\\ndef noise_image(img_arr, rng):\\n    vals = img_arr[img_arr != 0]\\n    return rng.normal(float(vals.mean()), float(vals.std()), size=img_arr.shape).astype(img_arr.dtype)\\n\\n\\ndef _region_metrics(mask, region, orig_voxels):\\n    n = int(mask.sum())\\n    inter = int(np.logical_and(mask, region).sum())\\n    cm, cr = centroid(mask), centroid(region)\\n    return {\\"n_pred_voxels\\": n, \\"size_vs_original\\": round(n / orig_voxels, 4) if orig_voxels else float(\\"nan\\"),\\n            \\"cov_of_noised_region\\": round(inter / max(int(region.sum()), 1), 4),\\n            \\"frac_of_mask_in_region\\": round(inter / n, 4) if n else 0.0,\\n            \\"centroid_offset_vox\\": round(float(np.linalg.norm(cm - cr)), 1) if cm is not None else float(\\"nan\\"),\\n            \\"empty\\": n == 0}\\n\\n\\ndef run_erasure(predictor, img_arr, label_vol, organs, ids, prompts, seed=0, return_slices=False):\\n    \\"\\"\\"Row 13. With return_slices=True it returns (rows, slices): for every prompt organ and condition one 2-D plane of the image\\n    the model received (float16) and of the predicted mask (uint8), plus the true organs on the same plane, so that the\\n    experiment can be drawn on real scan images. The plane cuts the volume across its thinnest axis at the named organ\'s centroid.\\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n    regions = {o: label_vol == ids[o] for o in organs}\\n    orig = predict_masks(predictor, img_arr, [prompts[o] for o in organs])\\n    orig_n = {o: int(orig[k].sum()) for k, o in enumerate(organs)}\\n    rows, slices = [], {}\\n    axis = int(np.argmin(label_vol.shape))                  # through-plane axis of the scan\\n    plane = {o: int(round(centroid(regions[o])[axis])) if regions[o].any() else label_vol.shape[axis] // 2 for o in organs}\\n\\n    def add(cond, o, mask, img):\\n        rows.append({\\"organ\\": o, \\"condition\\": cond, \\"prompt\\": prompts[o],\\n                     **_region_metrics(mask, regions[o], orig_n[o])})\\n        if return_slices:\\n            i = plane[o]\\n            slices[f\\"{o}|{cond}|image\\"] = np.take(img[0], i, axis=axis).astype(np.float16)\\n            slices[f\\"{o}|{cond}|mask\\"] = np.take(mask, i, axis=axis).astype(np.uint8)\\n\\n    for k, o in enumerate(organs):\\n        add(\\"original\\", o, orig[k], img_arr)\\n    for o in organs:                                        # only the named organ replaced by noise\\n        img = noise_region(img_arr, regions[o], rng)\\n        add(\\"noise_target\\", o, predict_masks(predictor, img, [prompts[o]])[0], img)\\n    for o in [x for x in organs if x.startswith(\\"left\\") and mirror(x) in organs]:   # both organs of the pair replaced by noise\\n        img = noise_region(noise_region(img_arr, regions[o], rng), regions[mirror(o)], rng)\\n        m = predict_masks(predictor, img, [prompts[o], prompts[mirror(o)]])\\n        add(\\"noise_pair\\", o, m[0], img)\\n        add(\\"noise_pair\\", mirror(o), m[1], img)\\n    nimg = noise_image(img_arr, rng)\\n    noisy = predict_masks(predictor, nimg, [prompts[o] for o in organs])\\n    for k, o in enumerate(organs):\\n        add(\\"noise_image\\", o, noisy[k], nimg)\\n    if return_slices:\\n        for o in organs:\\n            slices[f\\"{o}|axis\\"] = np.array(axis)\\n            slices[f\\"{o}|index\\"] = np.array(plane[o])\\n            for p_ in organs:\\n                slices[f\\"{o}|true|{p_}\\"] = np.take(regions[p_], plane[o], axis=axis).astype(np.uint8)\\n        return rows, slices\\n    return rows\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# rows 14-16 -- embedding intervention, cross-attention, layer-by-layer distance\\n# ------------------------------------------------------------------------------------------------\\nclass LayerDistanceRecorder:\\n    \\"\\"\\"Row 16: cosine distance and relative L2 distance between two prompts\' query states, per decoder layer\\n    (index 0 = the input state, 1..6 = after each layer), averaged over the patches that contain the organs\\n    and over all patches.\\"\\"\\"\\n\\n    def __init__(self, pair_defs, union_crop):\\n        self.pair_defs, self.union_crop = pair_defs, union_crop      # [(name, row_i, row_j)]\\n        self.union_padded = None\\n        self.cos, self.l2, self.flags = [], [], []\\n\\n    def reset(self):\\n        self.cos, self.l2, self.flags = [], [], []\\n\\n    def on_patch(self, slicer, states, attns, ctx):\\n        if self.union_padded is None:\\n            self.union_padded = np.zeros(ctx[\\"padded_shape\\"], bool)\\n            self.union_padded[ctx[\\"revert\\"][1:]] = self.union_crop\\n        self.flags.append(bool(self.union_padded[slicer[1:]].any()))\\n        c = np.zeros((len(self.pair_defs), len(states)))\\n        d = np.zeros_like(c)\\n        for pi, (_, i, j) in enumerate(self.pair_defs):\\n            for l, st in enumerate(states):\\n                a, b = st[i, 0].float(), st[j, 0].float()\\n                c[pi, l] = 1 - float(torch.nn.functional.cosine_similarity(a, b, dim=0))\\n                d[pi, l] = float(torch.linalg.norm(a - b) / (0.5 * (torch.linalg.norm(a) + torch.linalg.norm(b)) + 1e-8))\\n        self.cos.append(c)\\n        self.l2.append(d)\\n\\n    def rows(self):\\n        cos, l2, flags = np.array(self.cos), np.array(self.l2), np.array(self.flags)\\n        out = []\\n        for scope, sel in ((\\"organ_patches\\", flags), (\\"all_patches\\", np.ones_like(flags))):\\n            if not sel.any():\\n                continue\\n            for pi, (name, _, _) in enumerate(self.pair_defs):\\n                for l in range(cos.shape[2]):\\n                    out.append({\\"pair\\": name, \\"scope\\": scope, \\"layer\\": l, \\"n_patches\\": int(sel.sum()),\\n                                \\"cosine_distance\\": round(float(cos[sel, pi, l].mean()), 6),\\n                                \\"relative_l2_distance\\": round(float(l2[sel, pi, l].mean()), 6)})\\n        return out\\n\\n\\ndef _attention_rows(attn, organs, regions_crop, rl_axis):\\n    \\"\\"\\"Row 15. attn: (L, n_prompts, X, Y, Z) attention density (mean 1) in crop space; regions_crop: {organ: bool}.\\"\\"\\"\\n    dens_shape = attn.shape[2:]\\n    coords = torch.arange(dens_shape[rl_axis], dtype=torch.float32, device=attn.device)\\n    view = [1, 1, 1]\\n    view[rl_axis] = -1\\n    coords = coords.view(*view)\\n    rows, com_rows = [], []\\n    for l in range(attn.shape[0]):\\n        for p, prompt_organ in enumerate(organs):\\n            d = attn[l, p].float()\\n            total, mean_all = d.sum(), d.mean()\\n            for region_organ in organs:\\n                r = torch.from_numpy(regions_crop[region_organ]).to(d.device)\\n                if not bool(r.any()):\\n                    continue\\n                rows.append({\\"layer\\": l + 1, \\"prompt_organ\\": prompt_organ, \\"region_organ\\": region_organ,\\n                             \\"is_matched\\": region_organ == prompt_organ, \\"is_wrong_side\\": region_organ == mirror(prompt_organ),\\n                             \\"enrichment\\": round(float(d[r].mean() / mean_all), 4),\\n                             \\"mass_fraction\\": round(float(d[r].sum() / total), 6)})\\n            com = float((d * coords).sum() / total)\\n            named = centroid(regions_crop[prompt_organ])\\n            other = centroid(regions_crop[mirror(prompt_organ)]) if mirror(prompt_organ) in regions_crop else None\\n            if named is not None and other is not None:\\n                com_rows.append({\\"layer\\": l + 1, \\"prompt_organ\\": prompt_organ, \\"attention_centre_rl\\": round(com, 2),\\n                                 \\"named_organ_centre_rl\\": round(float(named[rl_axis]), 2),\\n                                 \\"other_side_organ_centre_rl\\": round(float(other[rl_axis]), 2),\\n                                 \\"closer_to_named_organ\\": abs(com - named[rl_axis]) < abs(com - other[rl_axis])})\\n    return rows, com_rows\\n\\n\\n_OOM = getattr(torch, \\"OutOfMemoryError\\", torch.cuda.OutOfMemoryError)      # older torch only has the cuda one\\n\\n\\ndef _run_chunked(predictor, data, emb_all, outputs, bbox, orig_shape, attn_factory, attn_rows, on_patch, reset, chunk=12):\\n    \\"\\"\\"Run the (many) outputs in chunks so the logits / decoder activations fit on the GPU. The attention and the layer\\n    recorder only need one pass, so they ride on the first chunk. A chunk that runs out of memory is retried with half the\\n    size. Returns (bool masks (n_out, X, Y, Z) in the original grid, attention or None).\\"\\"\\"\\n    import gc\\n    masks, attn, i = [], None, 0\\n    while i < len(outputs):\\n        part, first = outputs[i:i + chunk], i == 0\\n        oom = False\\n        try:\\n            logits, a = vm.run_outputs(predictor, data, emb_all, part,\\n                                       attn_acc_factory=attn_factory if first else None,\\n                                       attn_rows=attn_rows if first else None, on_patch=on_patch if first else None)\\n        except _OOM:\\n            oom = True\\n        if oom:                                   # clean up outside the except block so the failed frames are released\\n            gc.collect(); torch.cuda.empty_cache()\\n            if chunk == 1:\\n                raise RuntimeError(\\"out of GPU memory even with one output per pass\\")\\n            chunk = max(1, chunk // 2)\\n            if first:\\n                reset()\\n            continue\\n        masks.append(_to_full_masks(logits, bbox, orig_shape))\\n        del logits\\n        if first:\\n            attn = a.cpu() if a is not None else None\\n        i += len(part)\\n        gc.collect(); torch.cuda.empty_cache()\\n    return np.concatenate(masks), attn\\n\\n\\ndef run_interventions(predictor, img_arr, label_vol, organs, ids, prompts, rl_axis=2,\\n                      alphas=(-0.5, 0.25, 0.5, 0.75, 1.5), single_layers=range(6), from_layers=(0, 3),\\n                      state_layers=(0, 5), map_downsample=4, out_chunk=12):\\n    \\"\\"\\"Returns dict(row14=[...], row15=[...], row15_centre=[...], row16=[...], attention_maps={name: float16 array}).\\"\\"\\"\\n    pairs = [(o, mirror(o)) for o in organs if o.startswith(\\"left\\") and mirror(o) in organs]\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    emb_base = predictor.embed_text_prompts([prompts[o] for o in organs])          # (1, n, D)\\n    row_of = {o: i for i, o in enumerate(organs)}\\n    extra = []\\n    blend_row = {}\\n    for L, R in pairs:\\n        for a in alphas:\\n            blend_row[(L, a)] = len(organs) + len(extra)\\n            extra.append((1 - a) * emb_base[0, row_of[L]] + a * emb_base[0, row_of[R]])\\n    emb_all = torch.cat([emb_base, torch.stack(extra)[None]], dim=1) if extra else emb_base\\n\\n    outputs, meta = [], []\\n\\n    def add(name, row, ops, m):\\n        outputs.append({\\"name\\": name, \\"row\\": row, \\"ops\\": ops})\\n        meta.append(m)\\n\\n    for L, R in pairs:\\n        core = L.split(None, 1)[1]\\n        for o in (L, R):\\n            add(f\\"{o}|baseline\\", row_of[o], [], dict(pair=core, variant=\\"baseline\\", dst=o.split()[0], src=\\"\\", alpha=np.nan, layer=-1))\\n        for a in alphas:\\n            add(f\\"{L}|blend|{a}\\", blend_row[(L, a)], [], dict(pair=core, variant=\\"blend\\", dst=\\"left\\", src=\\"right\\", alpha=a, layer=-1))\\n        for dst, src in ((L, R), (R, L)):\\n            d, s = dst.split()[0], src.split()[0]\\n            for k in single_layers:\\n                add(f\\"{dst}|update|{k}\\", row_of[dst], [{\\"layer\\": k, \\"mode\\": \\"update\\", \\"src\\": row_of[src]}],\\n                    dict(pair=core, variant=\\"single_layer_update\\", dst=d, src=s, alpha=np.nan, layer=k))\\n            for k in from_layers:\\n                add(f\\"{dst}|from|{k}\\", row_of[dst], [{\\"layer\\": j, \\"mode\\": \\"update\\", \\"src\\": row_of[src]} for j in range(k, 6)],\\n                    dict(pair=core, variant=\\"updates_from_layer\\", dst=d, src=s, alpha=np.nan, layer=k))\\n            for k in state_layers:\\n                add(f\\"{dst}|state|{k}\\", row_of[dst], [{\\"layer\\": k, \\"mode\\": \\"state\\", \\"src\\": row_of[src]}],\\n                    dict(pair=core, variant=\\"whole_state_control\\", dst=d, src=s, alpha=np.nan, layer=k))\\n\\n    # ground truth in crop space (for the attention statistics and the recorder)\\n    crop_sl = tuple(slice(int(b[0]), int(b[1])) for b in bbox)\\n    label_crop = label_vol[crop_sl]\\n    regions_crop = {o: label_crop == ids[o] for o in organs}\\n    union_crop = np.zeros(label_crop.shape, bool)\\n    for r in regions_crop.values():\\n        union_crop |= r\\n    ref_pairs = []\\n    for L, R in pairs:\\n        others = [o for o in organs if o.split(None, 1)[1] != L.split(None, 1)[1] and o.startswith(\\"left\\")]\\n        ref_pairs.append((f\\"{L.split(None, 1)[1]}: left vs right\\", row_of[L], row_of[R]))\\n        if others:\\n            ref_pairs.append((f\\"{L.split(None, 1)[1]}: {L} vs {others[0]} (reference)\\", row_of[L], row_of[others[0]]))\\n    recorder = LayerDistanceRecorder(ref_pairs, union_crop)\\n\\n    masks, attn = _run_chunked(\\n        predictor, data, emb_all, outputs, bbox, orig_shape,\\n        attn_factory=lambda shape, n_layers, n_rows: vm.AttnAccumulator(shape, n_layers, n_rows, predictor.patch_size, predictor.device),\\n        attn_rows=len(organs), on_patch=recorder.on_patch, reset=recorder.reset, chunk=out_chunk)\\n    base_idx = {(m[\\"pair\\"], m[\\"dst\\"]): k for k, m in enumerate(meta) if m[\\"variant\\"] == \\"baseline\\"}\\n    rows14 = []\\n    for k, m in enumerate(meta):\\n        L = f\\"left {m[\'pair\']}\\"\\n        R = f\\"right {m[\'pair\']}\\"\\n        sc = vm.score_masks(masks[k], label_vol == ids[L], label_vol == ids[R])\\n        call = {\\"a\\": \\"left\\", \\"b\\": \\"right\\"}.get(sc[\\"call\\"], sc[\\"call\\"])\\n        moved = bool((m[\\"dst\\"] == \\"left\\" and call == \\"right\\") or (m[\\"dst\\"] == \\"right\\" and call == \\"left\\")) if m[\\"variant\\"] != \\"blend\\" else np.nan\\n        agree = float((masks[k] == masks[base_idx[(m[\\"pair\\"], m[\\"src\\"])]]).mean()) if m[\\"variant\\"] == \\"whole_state_control\\" else np.nan\\n        rows14.append({**m, \\"agreement_with_source_baseline\\": round(agree, 6) if agree == agree else agree,\\n                       \\"n_pred_voxels\\": sc[\\"n_pred_voxels\\"], \\"cov_left_organ\\": sc[\\"cov_a\\"], \\"cov_right_organ\\": sc[\\"cov_b\\"],\\n                       \\"dice_left_organ\\": sc[\\"dice_a\\"], \\"dice_right_organ\\": sc[\\"dice_b\\"], \\"call\\": call, \\"moved_to_source_side\\": moved})\\n    rows15, com15 = _attention_rows(attn, organs, regions_crop, rl_axis)\\n    maps = {}\\n    if map_downsample:\\n        pool = torch.nn.AvgPool3d(map_downsample, ceil_mode=True)\\n        for p, o in enumerate(organs):\\n            last = pool(attn[-1, p][None, None].float())[0, 0]\\n            mean = pool(attn[:, p].float().mean(0)[None, None])[0, 0]\\n            maps[f\\"{o}__last_layer\\"] = last.half().cpu().numpy()\\n            maps[f\\"{o}__mean_of_layers\\"] = mean.half().cpu().numpy()\\n    return {\\"row14\\": rows14, \\"row15\\": rows15, \\"row15_centre\\": com15, \\"row16\\": recorder.rows(), \\"attention_maps\\": maps}\\n\\n\\ndef validate_custom_loop(predictor, img_arr, prompts):\\n    \\"\\"\\"The custom sliding-window loop must reproduce the predictor\'s own masks; returns the voxel agreement.\\"\\"\\"\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    emb = predictor.embed_text_prompts(list(prompts))\\n    ref = predictor.predict_sliding_window_return_logits(data, emb)\\n    mine, _ = vm.run_outputs(predictor, data, emb, [{\\"name\\": str(i), \\"row\\": i, \\"ops\\": []} for i in range(emb.shape[1])])\\n    return float(((ref.float() > 0) == (mine.float() > 0)).float().mean())\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 17 -- voxel-grid occlusion Shapley map\\n# ------------------------------------------------------------------------------------------------\\ndef _block_mask(shape, block_slicer):\\n    m = np.zeros(shape, dtype=bool)\\n    m[block_slicer] = True\\n    return m\\n\\n\\ndef _bbox_str(block_slicer):\\n    return \\"x\\".join(f\\"{s.start}:{s.stop}\\" for s in block_slicer)\\n\\n\\ndef run_grid_occlusion(predictor, img_arr, label_vol, organ, ids, prompt, grid=2, pad=10, seed=0, return_map=False):\\n    \\"\\"\\"Row 17. Tile a padded box around `organ`\'s ground-truth region into `grid`**3 blocks (default\\n    8 blocks -> 2**8=256 coalitions, the same affordability argument already used for row 5\'s exact\\n    word-Shapley: 2**8 is cheap to enumerate exactly). Each block is independently noised/kept\\n    (reusing `noise_region`), `f_dice_gt` is scored per coalition, and `shapley_values` (same math as\\n    the word-attribution rows) turns that into one Shapley credit per block -- the image-region\\n    analogue of row 5\'s word table. Returns rows with columns: organ, block_id, block_bbox, shapley_value.\\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n    region = label_vol == ids[organ]\\n    if not region.any():\\n        return []\\n    idx = np.array(np.nonzero(region))\\n    mins = np.clip(idx.min(axis=1) - pad, 0, None)\\n    maxs = np.minimum(idx.max(axis=1) + pad + 1, np.array(label_vol.shape))\\n    edges = [np.linspace(mins[a], maxs[a], grid + 1).round().astype(int) for a in range(3)]\\n    blocks = [(slice(edges[0][i], edges[0][i + 1]), slice(edges[1][j], edges[1][j + 1]), slice(edges[2][k], edges[2][k + 1]))\\n              for i in range(grid) for j in range(grid) for k in range(grid)]\\n    block_ids = list(range(len(blocks)))\\n\\n    v = {}\\n    for key, present in all_role_combos(block_ids):\\n        img = img_arr\\n        for b in block_ids:\\n            if not present[b]:\\n                img = noise_region(img, _block_mask(label_vol.shape, blocks[b]), rng)\\n        mask = predict_masks(predictor, img, [prompt])[0]\\n        v[key] = vm.score_masks(mask, region, region)[\\"dice_a\\"]\\n    phi = shapley_values(v, block_ids)\\n    rows = [{\\"organ\\": organ, \\"block_id\\": b, \\"block_bbox\\": _bbox_str(blocks[b]), \\"shapley_value\\": round(phi[b], 6)}\\n            for b in block_ids]\\n    if not return_map:\\n        return rows\\n    voxel_map = np.full(label_vol.shape, np.nan, dtype=np.float32)\\n    for b in block_ids:\\n        voxel_map[blocks[b]] = phi[b]\\n    return rows, voxel_map\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 18 -- per-layer attention/KL divergence against the ground-truth mask\\n# ------------------------------------------------------------------------------------------------\\ndef run_attention_kl(predictor, img_arr, label_vol, organs, ids, prompts,\\n                      generic_prompt=\\"Segment the anatomy in this image.\\", out_chunk=12,\\n                      return_maps=False, map_downsample=4):\\n    \\"\\"\\"Row 18 (extends row 15\'s attention pipeline). For every organ and decoder layer, computes\\n    KL(GT mask distribution || cross-attention density distribution) over voxels -- the diagnostic\\n    used by the VGMED/VGRefine paper (arXiv 2603.14323) to find which layer\'s attention best matches\\n    the clinically relevant region. Also reports the same KL after dividing out a generic, non-spatial\\n    prompt\'s attention map (their reference-prompt normalization), which isolates prompt-specific\\n    attention from wherever VoxTell looks by default. Returns rows: organ, layer, kl_divergence,\\n    kl_divergence_vs_generic_normalized (lower = attention better matches the GT region).\\n\\n    return_maps=True also returns a dict (same convention as `run_interventions`\'s `attention_maps`):\\n    per organ, the last-layer and mean-of-layers attention density, downsampled by `map_downsample`\\n    -- an actual spatial map to overlay on the image/GT (row 0), not just the scalar KL above.\\"\\"\\"\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    emb_real = predictor.embed_text_prompts([prompts[o] for o in organs])\\n    emb_generic = predictor.embed_text_prompts([generic_prompt])\\n    emb_all = torch.cat([emb_real, emb_generic], dim=1)\\n    outputs = [{\\"name\\": o, \\"row\\": i, \\"ops\\": []} for i, o in enumerate(organs)]\\n    outputs.append({\\"name\\": \\"__generic__\\", \\"row\\": len(organs), \\"ops\\": []})\\n\\n    crop_sl = tuple(slice(int(b[0]), int(b[1])) for b in bbox)\\n    label_crop = label_vol[crop_sl]\\n    regions_crop = {o: label_crop == ids[o] for o in organs}\\n\\n    _, attn = _run_chunked(\\n        predictor, data, emb_all, outputs, bbox, orig_shape,\\n        attn_factory=lambda shape, n_layers, n_rows: vm.AttnAccumulator(shape, n_layers, n_rows, predictor.patch_size, predictor.device),\\n        attn_rows=len(organs) + 1, on_patch=None, reset=lambda: None, chunk=out_chunk)\\n\\n    eps = 1e-8\\n\\n    def kl(p_norm, q_norm):\\n        return float((p_norm * (torch.log(p_norm.clamp_min(eps)) - torch.log(q_norm.clamp_min(eps)))).sum())\\n\\n    rows = []\\n    for p, organ in enumerate(organs):\\n        gt = torch.from_numpy(regions_crop[organ].astype(np.float32))\\n        if float(gt.sum()) == 0:\\n            continue\\n        gt_norm = gt / gt.sum()\\n        for l in range(attn.shape[0]):\\n            d = attn[l, p].float()\\n            d_norm = d / d.sum().clamp_min(eps)\\n            g = attn[l, len(organs)].float()\\n            g_norm = g / g.sum().clamp_min(eps)\\n            ratio = d_norm / g_norm.clamp_min(eps)\\n            ratio_norm = ratio / ratio.sum().clamp_min(eps)\\n            rows.append({\\"organ\\": organ, \\"layer\\": l + 1,\\n                         \\"kl_divergence\\": round(kl(gt_norm, d_norm), 6),\\n                         \\"kl_divergence_vs_generic_normalized\\": round(kl(gt_norm, ratio_norm), 6)})\\n    if not return_maps:\\n        return rows\\n    maps = {}\\n    pool = torch.nn.AvgPool3d(map_downsample, ceil_mode=True)\\n    for p, organ in enumerate(organs):\\n        last = pool(attn[-1, p][None, None].float())[0, 0]\\n        mean = pool(attn[:, p].float().mean(0)[None, None])[0, 0]\\n        maps[f\\"{organ}__last_layer\\"] = last.half().cpu().numpy()\\n        maps[f\\"{organ}__mean_of_layers\\"] = mean.half().cpu().numpy()\\n    return rows, maps\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 21 -- joint image-vs-word Shapley (3 roles: image, organ word, position word)\\n# ------------------------------------------------------------------------------------------------\\ndef _build_role_prompt(organ_name, organ_word_present, position_word_present):\\n    position, core = None, organ_name\\n    parts = organ_name.split(None, 1)\\n    if len(parts) == 2 and parts[0].lower() in (\\"left\\", \\"right\\"):\\n        position, core = parts[0].lower(), parts[1]\\n    words = [\\"Segment\\", \\"the\\"]\\n    if position_word_present and position:\\n        words.append(position)\\n    words.append(core if organ_word_present else \\"structure\\")\\n    words += [\\"in\\", \\"this\\", \\"image\\"]\\n    s = \\" \\".join(words)\\n    return s[0].upper() + s[1:] + \\".\\"\\n\\n\\ndef _crop_patch_around(data, centre_vox, patch_size):\\n    \\"\\"\\"Crop a `patch_size`-shaped window of `data` (C, X, Y, Z) centred on `centre_vox`, clamped to\\n    stay inside the array. `data` is `predictor.preprocess(img_arr)`\'s own output -- a torch tensor,\\n    not a numpy array (confirmed by every other row\'s usage, e.g. it\'s passed straight into\\n    `predictor.predict_sliding_window_return_logits`/`vm.run_outputs` elsewhere in this file) -- so\\n    this works natively in torch, not `np.zeros`/`torch.from_numpy`, which both raise on a tensor\\n    input (`np.zeros(..., dtype=<torch dtype>)` cannot interpret a torch dtype; `torch.from_numpy`\\n    requires an actual numpy array). Still accepts a plain numpy array too, for any future caller.\\n    Returns (patch (1, C, *patch_size), offset (3,) -- the crop\'s start index, so results can be\\n    mapped back into `data`\'s coordinate frame if needed).\\"\\"\\"\\n    is_tensor = torch.is_tensor(data)\\n    shape = np.array(data.shape[1:])\\n    ps = np.array(patch_size)\\n    start = np.clip(np.round(np.asarray(centre_vox) - ps / 2).astype(int), 0, np.maximum(shape - ps, 0))\\n    end = np.minimum(start + ps, shape)\\n    start = np.maximum(end - ps, 0)  # re-clamp if the patch is larger than the volume near an edge\\n    sl = tuple(slice(int(s), int(s) + int(p)) for s, p in zip(start, ps))\\n    patch = data[(slice(None), *sl)]\\n    if tuple(patch.shape[1:]) != tuple(int(p) for p in ps):  # volume smaller than patch_size on some axis -- pad with zeros\\n        if is_tensor:\\n            padded = torch.zeros((data.shape[0], *(int(p) for p in ps)), dtype=data.dtype, device=data.device)\\n        else:\\n            padded = np.zeros((data.shape[0], *(int(p) for p in ps)), dtype=data.dtype)\\n        padded[(slice(None), *tuple(slice(0, s) for s in patch.shape[1:]))] = patch\\n        patch = padded\\n    return patch[None], start\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 19 -- gradient-based saliency (vanilla gradient + Grad-CAM), one patch around the organ\\n# ------------------------------------------------------------------------------------------------\\ndef run_gradient_saliency(predictor, img_arr, label_vol, organ, ids, prompt, return_maps=False):\\n    \\"\\"\\"Row 19. Crops one patch_size window of the preprocessed image centred on the organ\'s GT\\n    centroid (cheap: one encoder + one backward pass, vs. row 17\'s full sliding-window Shapley\\n    enumeration -- the same LIME-to-Shapley relationship, just for images), computes vanilla-gradient\\n    and Grad-CAM saliency (`vm.gradient_saliency`), and summarizes it against the GT mask inside that\\n    same patch. MUST be called outside any `torch.no_grad()`/`inference_mode()` context.\\n    Returns a single-row dict: organ, mean_saliency_in_organ, mean_saliency_outside_organ,\\n    saliency_enrichment, cam_mean_in_organ, cam_mean_outside_organ.\\"\\"\\"\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    crop_sl = tuple(slice(int(b[0]), int(b[1])) for b in bbox)\\n    label_crop = label_vol[crop_sl]\\n    region = label_crop == ids[organ]\\n    c = centroid(region)\\n    if c is None:\\n        return None\\n    patch, offset = _crop_patch_around(data, c, predictor.patch_size)\\n    patch_t = (patch if torch.is_tensor(patch) else torch.from_numpy(patch)).to(predictor.device)\\n    emb = predictor.embed_text_prompts([prompt])[0].to(predictor.device)  # (1, D)\\n    saliency, cam = vm.gradient_saliency(predictor.network, patch_t, row=0, prompts_embed=emb)\\n    end = offset + np.array(predictor.patch_size)\\n    region_in_patch = np.zeros_like(saliency, dtype=bool)\\n    rs = tuple(slice(max(int(o), 0), min(int(e), int(l))) for o, e, l in zip(offset, end, label_crop.shape))\\n    ps = tuple(slice(0, s.stop - s.start) for s in rs)\\n    region_in_patch[ps] = region[rs]\\n    if not region_in_patch.any():\\n        return None\\n    cam_up = vm.grad_cam_upsample(cam, predictor.patch_size)\\n    out_sal_in, out_sal_out = float(saliency[region_in_patch].mean()), float(saliency[~region_in_patch].mean())\\n    cam_in, cam_out = float(cam_up[region_in_patch].mean()), float(cam_up[~region_in_patch].mean())\\n    row = {\\"organ\\": organ, \\"mean_saliency_in_organ\\": round(out_sal_in, 6),\\n           \\"mean_saliency_outside_organ\\": round(out_sal_out, 6),\\n           \\"saliency_enrichment\\": round(out_sal_in / max(out_sal_out, 1e-8), 4),\\n           \\"cam_mean_in_organ\\": round(cam_in, 6), \\"cam_mean_outside_organ\\": round(cam_out, 6),\\n           \\"cam_enrichment\\": round(cam_in / max(cam_out, 1e-8), 4)}\\n    if not return_maps:\\n        return row\\n    # place the patch\'s saliency/CAM back into the crop\'s own coordinate frame, for plotting\\n    # against the same image/GT the rest of this module works in\\n    full_saliency = np.zeros(label_crop.shape, dtype=np.float32)\\n    full_cam = np.zeros(label_crop.shape, dtype=np.float32)\\n    full_saliency[rs] = saliency[ps]\\n    full_cam[rs] = cam_up[ps]\\n    return row, full_saliency, full_cam\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 20 -- fused spatial feature map, matched vs. mismatched prompt (resolves plan row 1)\\n# ------------------------------------------------------------------------------------------------\\ndef run_activation_overlap(predictor, img_arr, label_vol, organ, ids, prompts, stage=-1,\\n                            return_maps=False, map_downsample=4):\\n    \\"\\"\\"Row 20 -- a deeper repeat of plan row 1 (already run using the final pre-sigmoid logits as\\n    its \\"internal representation\\") at a genuine intermediate layer. For `organ`\'s matched prompt\\n    and two mismatches (wrong side via `mirror`, wrong organ via a different organ in `ids`),\\n    captures VoxTellDecoder\'s spatial `fusion_features` (`vm.FusedSpatialCapture`) and measures IoU\\n    between its highest-activation region and the GT bbox mask (`vm.activation_overlap_iou`).\\n    `stage`: which captured stage to read (-1 = highest resolution / last stage). Returns up to 3\\n    rows: organ, condition, stage, iou_with_gt.\\n\\n    return_maps=True also returns a dict {condition: array}, the fused-feature map itself, upsampled\\n    to the crop\'s coordinate frame (same frame as row 0\'s masks) and then downsampled by\\n    `map_downsample` for storage -- an actual spatial map to overlay, not just the scalar IoU.\\"\\"\\"\\n    gt = label_vol == ids[organ]\\n    if not gt.any():\\n        return ([], {}) if return_maps else []\\n    other_organ = next((o for o in ids if o != organ and o != mirror(organ)), None)\\n    conditions = [(\\"matched\\", prompts[organ])]\\n    if mirror(organ) in prompts:\\n        conditions.append((\\"mismatched_side\\", prompts[mirror(organ)]))\\n    if other_organ is not None:\\n        conditions.append((\\"mismatched_organ\\", prompts[other_organ]))\\n\\n    rows, maps = [], {}\\n    pool = torch.nn.AvgPool3d(map_downsample, ceil_mode=True)\\n    for cond, prompt in conditions:\\n        cap = vm.FusedSpatialCapture(predictor.network.decoder)\\n        cap.attach()\\n        try:\\n            data, bbox, orig_shape = predictor.preprocess(img_arr)\\n            emb = predictor.embed_text_prompts([prompt])\\n            with torch.no_grad():\\n                predictor.predict_sliding_window_return_logits(data, emb)\\n        finally:\\n            cap.detach()\\n        if not cap.maps:\\n            continue\\n        stage_idx = sorted(cap.maps)[stage]\\n        crop_sl = tuple(slice(int(b[0]), int(b[1])) for b in bbox)\\n        gt_crop = gt[crop_sl]\\n        iou = vm.activation_overlap_iou(cap.maps[stage_idx][0], gt_crop)\\n        rows.append({\\"organ\\": organ, \\"condition\\": cond, \\"stage\\": stage_idx, \\"iou_with_gt\\": round(iou, 4)})\\n        if return_maps:\\n            m = cap.maps[stage_idx][0].float()\\n            if m.ndim == 4:\\n                m = m.abs().mean(dim=0)\\n            up = torch.nn.functional.interpolate(m[None, None], size=tuple(gt_crop.shape), mode=\'trilinear\', align_corners=False)\\n            maps[f\\"{organ}__{cond}\\"] = pool(up)[0, 0].half().cpu().numpy()\\n    return (rows, maps) if return_maps else rows\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 0 -- baseline prediction + GT export, for plotting (image + GT + predicted mask overlays)\\n# ------------------------------------------------------------------------------------------------\\ndef export_baseline_masks(predictor, img_arr, label_vol, organs, ids, prompts, image_path=None, label_path=None):\\n    \\"\\"\\"Not a plan row of its own -- a shared plotting export run once per subject, before any of\\n    rows 12-21. For every organ: the full-prompt predicted mask and the GT mask, both RLE-encoded\\n    (`encode_mask_rle`) so the complete 3D mask -- not just a summary metric -- can be reloaded and\\n    overlaid on the source image slice-by-slice later (`decode_mask_rle` + `image_path`/`label_path`\\n    from the manifest). Returns one row per organ: organ, prompt, image_path, label_path,\\n    volume_shape, dice, n_pred_voxels, pred_centroid_vox_{0,1,2}, gt_centroid_vox_{0,1,2},\\n    pred_mask_rle_b64, gt_mask_rle_b64.\\"\\"\\"\\n    masks = predict_masks(predictor, img_arr, [prompts[o] for o in organs])\\n    rows = []\\n    for k, o in enumerate(organs):\\n        gt = label_vol == ids[o]\\n        pred = masks[k]\\n        pc, gc = centroid(pred), centroid(gt)\\n        rows.append({\\n            \\"organ\\": o, \\"prompt\\": prompts[o],\\n            \\"image_path\\": image_path, \\"label_path\\": label_path,\\n            \\"volume_shape\\": \\"x\\".join(str(s) for s in label_vol.shape),\\n            \\"dice\\": round(vm.score_masks(pred, gt, gt)[\\"dice_a\\"], 4),\\n            \\"n_pred_voxels\\": int(pred.sum()), \\"n_gt_voxels\\": int(gt.sum()),\\n            \\"pred_centroid_vox_0\\": round(float(pc[0]), 2) if pc is not None else None,\\n            \\"pred_centroid_vox_1\\": round(float(pc[1]), 2) if pc is not None else None,\\n            \\"pred_centroid_vox_2\\": round(float(pc[2]), 2) if pc is not None else None,\\n            \\"gt_centroid_vox_0\\": round(float(gc[0]), 2) if gc is not None else None,\\n            \\"gt_centroid_vox_1\\": round(float(gc[1]), 2) if gc is not None else None,\\n            \\"gt_centroid_vox_2\\": round(float(gc[2]), 2) if gc is not None else None,\\n            \\"pred_mask_rle_b64\\": encode_mask_rle(pred),\\n            \\"gt_mask_rle_b64\\": encode_mask_rle(gt),\\n        })\\n    return rows\\n\\n\\ndef run_joint_image_word_shapley(predictor, img_arr, label_vol, organ, ids, seed=0, return_coalitions=False):\\n    \\"\\"\\"Row 21. Three roles -- image (intact / noised over the organ\'s own GT region, reusing row 13\'s\\n    `noise_region`), organ_word, position_word (reusing the same prompt-building logic as rows 5-11) --\\n    give 2**3=8 coalitions, scored by both f_dice_gt and f_dice_self and fed into the same\\n    `shapley_values` used for row 17 and the word-attribution rows. This is the single number that\\n    puts image contribution and word contribution on one common scale. Returns rows: organ, role,\\n    metric, shapley_value (already in Dice-score units: shapley_value is literally \\"how many Dice\\n    points this role is worth, averaged fairly over every order the roles could be added in\\").\\n\\n    return_coalitions=True additionally returns the literal, un-derived numbers the Shapley credit\\n    above is computed FROM: the actual Dice score for each of the 8 present/absent combinations\\n    (e.g. \\"if the model only gets the image, no words at all, what Dice does it get?\\"). These are\\n    more intuitive to read directly but don\'t have the Shapley math\'s fairness guarantees (e.g. they\\n    don\'t sum to the total the way the credits do) -- returns (summary_rows, coalition_rows), where\\n    coalition_rows has columns: organ, roles_present, image, organ_word, position_word, dice_gt,\\n    dice_self.\\"\\"\\"\\n    rng = np.random.default_rng(seed)\\n    gt = label_vol == ids[organ]\\n    roles = [\\"image\\", \\"organ_word\\", \\"position_word\\"]\\n    masks = {}\\n    for key, present in all_role_combos(roles):\\n        img = img_arr if present[\\"image\\"] else noise_region(img_arr, gt, rng)\\n        prompt = _build_role_prompt(organ, present[\\"organ_word\\"], present[\\"position_word\\"])\\n        masks[key] = predict_masks(predictor, img, [prompt])[0]\\n    base_mask = masks[frozenset(roles)]\\n    v_gt = {k: vm.score_masks(m, gt, gt)[\\"dice_a\\"] for k, m in masks.items()}\\n    v_self = {k: vm.score_masks(m, base_mask, base_mask)[\\"dice_a\\"] for k, m in masks.items()}\\n    phi_gt, phi_self = shapley_values(v_gt, roles), shapley_values(v_self, roles)\\n    rows = []\\n    for r in roles:\\n        rows.append({\\"organ\\": organ, \\"role\\": r, \\"metric\\": \\"f_dice_gt\\", \\"shapley_value\\": round(phi_gt[r], 6)})\\n        rows.append({\\"organ\\": organ, \\"role\\": r, \\"metric\\": \\"f_dice_self\\", \\"shapley_value\\": round(phi_self[r], 6)})\\n    if not return_coalitions:\\n        return rows\\n    coalition_rows = []\\n    for key in v_gt:\\n        coalition_rows.append({\\n            \\"organ\\": organ,\\n            \\"roles_present\\": \\"+\\".join(sorted(key)) if key else \\"none\\",\\n            \\"image\\": \\"image\\" in key, \\"organ_word\\": \\"organ_word\\" in key, \\"position_word\\": \\"position_word\\" in key,\\n            \\"dice_gt\\": round(v_gt[key], 4), \\"dice_self\\": round(v_self[key], 4),\\n        })\\n    return rows, coalition_rows\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 22 -- linear probing: is left/right linearly DECODABLE at each decoder layer, not just different?\\n# ------------------------------------------------------------------------------------------------\\ndef _home_patch_indices(predictor, data, bbox, label_vol, organs, ids):\\n    \\"\\"\\"For every organ, the sliding-window patch index (into the SAME slicer list `run_outputs`\\n    builds internally, in the same order -- both are pure functions of `data.shape` and\\n    `predictor.patch_size`, so the index computed here lines up with `on_patch`\'s call order)\\n    whose block contains that organ\'s own ground-truth centroid, or is nearest to it if no single\\n    block contains it. This is \\"the patch that actually produces the final mask there\\" -- the\\n    representative sliding-window pass for that organ, as opposed to averaging over the whole,\\n    mostly-irrelevant, volume.\\"\\"\\"\\n    from acvl_utils.cropping_and_padding.padding import pad_nd_image\\n    data_p, revert = pad_nd_image(data, predictor.patch_size, \'constant\', {\'value\': 0}, True, None)\\n    slicers = predictor._internal_get_sliding_window_slicers(data_p.shape[1:])\\n    crop_sl = tuple(slice(int(b[0]), int(b[1])) for b in bbox)\\n    label_crop = label_vol[crop_sl]\\n    pad_before = [s.start for s in revert[1:]]\\n    home = {}\\n    for organ in organs:\\n        c = centroid(label_crop == ids[organ])\\n        if c is None:\\n            continue\\n        pt = [float(c[a]) + pad_before[a] for a in range(3)]\\n        best_i, best_score = None, None\\n        for i, slicer in enumerate(slicers):\\n            sl = slicer[1:]\\n            inside = all(sl[a].start <= pt[a] < sl[a].stop for a in range(3))\\n            centre = [(sl[a].start + sl[a].stop) / 2 for a in range(3)]\\n            dist2 = sum((pt[a] - centre[a]) ** 2 for a in range(3))\\n            score = (0 if inside else 1, dist2)\\n            if best_score is None or score < best_score:\\n                best_i, best_score = i, score\\n        home[organ] = best_i\\n    return home\\n\\n\\ndef run_laterality_probing(predictor, img_arr, label_vol, organs, ids, prompts):\\n    \\"\\"\\"Row 22. Trains a linear probe (logistic regression) at every decoder layer -- state 0 is the\\n    projected text embedding itself (`tgt0`, before any cross-attention), states 1-6 are after each\\n    decoder layer -- to predict left vs. right from the hidden state ALONE, independent of what mask\\n    the model actually outputs. This is a stricter question than row 16\'s cosine distance: distance\\n    can stay small while a linear probe still finds a perfectly usable separating direction (distance\\n    measures \\"how different two specific states are\\", the probe measures \\"is side actually recoverable\\n    from this representation by a simple linear read-out\\"). If a layer\'s probe is near-perfect but the\\n    final mask still gets the side wrong, that is direct evidence the DECODER discards information\\n    that was already there, not that the information was never computed (an alignment failure would\\n    instead show a probe that stays near-chance at every layer).\\n\\n    The hidden state for each organ is read from its own \\"home patch\\" (`_home_patch_indices` above) --\\n    the one sliding-window patch that actually contains that organ\'s ground-truth centroid -- not\\n    averaged across the whole volume, most of which is irrelevant to that organ\'s prediction.\\n\\n    Small-n caveat (stated explicitly in the returned rows via `n_pairs`): with a single subject and\\n    typically single-digit bilateral organ pairs, cross-validation here is leave-one-ORGAN-pair-out\\n    (both the left and right instance of a pair are held out together, so the probe is never trained\\n    and tested on the same organ identity) -- indicative, not a robust estimate. Returns rows: layer,\\n    cv_accuracy, train_accuracy, n_pairs.\\"\\"\\"\\n    from sklearn.linear_model import LogisticRegression\\n    from sklearn.model_selection import LeaveOneGroupOut\\n\\n    data, bbox, orig_shape = predictor.preprocess(img_arr)\\n    home = _home_patch_indices(predictor, data, bbox, label_vol, organs, ids)\\n    cores = sorted({o.split(None, 1)[1] for o in organs if o.split(None, 1)[0] in (\\"left\\", \\"right\\")})\\n    pairs = [c for c in cores if f\\"left {c}\\" in home and f\\"right {c}\\" in home]\\n    if len(pairs) < 2:\\n        return []\\n\\n    wanted = {}\\n    for i, c in enumerate(pairs):\\n        wanted[f\\"left {c}\\"] = 2 * i\\n        wanted[f\\"right {c}\\"] = 2 * i + 1\\n    ordered_organs = sorted(wanted, key=wanted.get)\\n    emb = predictor.embed_text_prompts([prompts[o] for o in ordered_organs])\\n    target_patches = {home[o]: o for o in wanted}\\n\\n    captured = {}\\n    counter = {\\"i\\": 0}\\n\\n    def on_patch(slicer, states, attns, ctx):\\n        organ = target_patches.get(counter[\\"i\\"])\\n        counter[\\"i\\"] += 1\\n        if organ is not None:\\n            r = wanted[organ]\\n            captured[organ] = [s[r, 0].float().cpu().numpy() for s in states]\\n\\n    vm.run_outputs(predictor, data, emb, [{\\"name\\": o, \\"row\\": wanted[o], \\"ops\\": []} for o in ordered_organs],\\n                    on_patch=on_patch)\\n\\n    organs_order = [o for o in ordered_organs if o in captured]\\n    if len(organs_order) < 4:  # need at least 2 complete pairs to do leave-one-pair-out at all\\n        return []\\n    n_layers = len(captured[organs_order[0]])\\n    y = np.array([1 if o.startswith(\\"right\\") else 0 for o in organs_order])\\n    groups = np.array([o.split(None, 1)[1] for o in organs_order])\\n\\n    rows = []\\n    for layer in range(n_layers):\\n        X = np.stack([captured[o][layer] for o in organs_order])\\n        logo = LeaveOneGroupOut()\\n        correct, total = 0, 0\\n        for tr, te in logo.split(X, y, groups=groups):\\n            if len(set(y[tr])) < 2:\\n                continue\\n            clf = LogisticRegression(max_iter=2000).fit(X[tr], y[tr])\\n            correct += int((clf.predict(X[te]) == y[te]).sum())\\n            total += len(te)\\n        cv_acc = correct / total if total else float(\\"nan\\")\\n        clf_full = LogisticRegression(max_iter=2000).fit(X, y)\\n        train_acc = float((clf_full.predict(X) == y).mean())\\n        rows.append({\\"layer\\": layer, \\"cv_accuracy\\": round(cv_acc, 4) if total else None,\\n                     \\"train_accuracy\\": round(train_acc, 4), \\"n_pairs\\": len(pairs)})\\n    return rows\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 23 -- embedding geometry: one consistent \\"left -> right\\" direction, or a different one per organ?\\n# ------------------------------------------------------------------------------------------------\\ndef run_embedding_geometry(predictor, organs, prompts):\\n    \\"\\"\\"Row 23. Does the model use ONE consistent \\"left -> right\\" direction across every organ, the\\n    way a genuinely spatial/relational representation would (the classic word-vector-arithmetic\\n    test, e.g. \\"king - man + woman\\"), or does each organ get its own, unrelated left/right\\n    direction (meaning \\"left\\" is re-derived per organ, not a shared, composable concept)? For\\n    every bilateral pair, computes `embed(\\"right X\\") - embed(\\"left X\\")` in the raw text-encoder\\n    embedding space (the same embedding rows 2-4 and row 22\'s layer-0 state are built from, before\\n    `project_text_embed`). Pure text-embedding arithmetic -- no image, no sliding window, no model\\n    forward pass beyond the text encoder: the cheapest row in this whole module.\\n\\n    Returns one schema for every row (fields not applicable to a row\'s `kind` are None), two kinds:\\n      kind=\'pair\'    -- organ_a, organ_b, cosine_similarity between their two diff vectors\\n      kind=\'summary\' -- mean_cosine_similarity (mean of all pairwise similarities), coherence_ratio\\n                        = ||mean(diff vectors)|| / mean(||diff vector||): 1.0 means every organ\'s\\n                        \\"left->right\\" step points the same way (one reusable direction), 0.0 means\\n                        the directions are unrelated/random per organ, n_organs.\\"\\"\\"\\n    cores = sorted({o.split(None, 1)[1] for o in organs if o.split(None, 1)[0] in (\\"left\\", \\"right\\")})\\n    cores = [c for c in cores if f\\"left {c}\\" in prompts and f\\"right {c}\\" in prompts]\\n    if len(cores) < 2:\\n        return []\\n    left_emb = predictor.embed_text_prompts([prompts[f\\"left {c}\\"] for c in cores])[0]\\n    right_emb = predictor.embed_text_prompts([prompts[f\\"right {c}\\"] for c in cores])[0]\\n    diffs = (right_emb - left_emb).float().cpu().numpy()\\n    norms = np.linalg.norm(diffs, axis=1)\\n    unit = diffs / np.clip(norms, 1e-8, None)[:, None]\\n    sim = unit @ unit.T\\n\\n    rows = []\\n    for i in range(len(cores)):\\n        for j in range(i + 1, len(cores)):\\n            rows.append({\\"kind\\": \\"pair\\", \\"organ_a\\": cores[i], \\"organ_b\\": cores[j],\\n                         \\"cosine_similarity\\": round(float(sim[i, j]), 4),\\n                         \\"mean_cosine_similarity\\": None, \\"coherence_ratio\\": None, \\"n_organs\\": None})\\n    mean_cos = float(sim[np.triu_indices(len(cores), k=1)].mean())\\n    coherence = float(np.linalg.norm(diffs.mean(axis=0)) / max(float(norms.mean()), 1e-8))\\n    rows.append({\\"kind\\": \\"summary\\", \\"organ_a\\": None, \\"organ_b\\": None, \\"cosine_similarity\\": None,\\n                 \\"mean_cosine_similarity\\": round(mean_cos, 4), \\"coherence_ratio\\": round(coherence, 4),\\n                 \\"n_organs\\": len(cores)})\\n    return rows\\n\\n\\n# ------------------------------------------------------------------------------------------------\\n# row 24 -- cross-attention head ablation: which specific head, not just which layer\\n# ------------------------------------------------------------------------------------------------\\ndef run_attention_head_ablation(predictor, img_arr, label_vol, organs, ids, prompts, layers=None):\\n    \\"\\"\\"Row 24. Zero-ablates each cross-attention HEAD, one at a time (`vm.HeadAblation`), at each\\n    layer in `layers` (default: the LAST layer only -- cheapest, and closest to the final decision;\\n    pass e.g. `layers=range(6)` for every layer, at 6x the cost: n_organs x n_layers x n_heads full\\n    sliding-window predictions, the same affordability trade-off as row 17\'s block count), and\\n    re-predicts the FULL mask through the predictor\'s own public API. The ablation hook operates at\\n    the `nn.Module` level, so it affects the public `predict_sliding_window_return_logits` path\\n    identically to the custom loop -- no custom inference loop needed here, unlike rows 14-16.\\n\\n    MUST call `vm.validate_head_ablation_noop` first and confirm it returns 1.0 (up to floating-point\\n    rounding) -- see this module\'s own validation cell in the notebook.\\n\\n    For every organ, layer and head: re-predicts the mask with that head zeroed, scores Dice against\\n    GT and against the organ\'s own UNABLATED baseline mask (f_dice_self -- did ablating this head\\n    change the answer AT ALL), and the same `call` classification rows 12/13 use (does the mask still\\n    land on this organ, on its mirror, on both, or neither) against {this organ, its mirror}. A head\\n    whose ablation collapses dice_vs_gt or flips `call` toward the mirror organ is doing real\\n    laterality work; a head whose ablation changes nothing (dice_vs_baseline ~= 1) is not involved in\\n    this decision at all. Returns rows: organ, layer, head, n_heads, dice_vs_gt, dice_vs_baseline, call.\\"\\"\\"\\n    num_layers = len(predictor.network.transformer_decoder.layers)\\n    layers = list(layers) if layers is not None else [num_layers - 1]\\n    n_heads = vm.num_cross_attn_heads(predictor.network.transformer_decoder.layers)\\n\\n    baselines = {o: predict_masks(predictor, img_arr, [prompts[o]])[0] for o in organs}\\n    rows = []\\n    for organ in organs:\\n        gt = label_vol == ids[organ]\\n        gt_mirror = label_vol == ids[mirror(organ)] if mirror(organ) in ids else np.zeros_like(gt)\\n        base_mask = baselines[organ]\\n        for layer in layers:\\n            for head in range(n_heads):\\n                ablation = vm.HeadAblation(predictor.network.transformer_decoder.layers, [layer], [head])\\n                ablation.attach()\\n                try:\\n                    mask = predict_masks(predictor, img_arr, [prompts[organ]])[0]\\n                finally:\\n                    ablation.detach()\\n                scored = vm.score_masks(mask, gt, gt_mirror)\\n                rows.append({\\n                    \\"organ\\": organ, \\"layer\\": layer, \\"head\\": head, \\"n_heads\\": n_heads,\\n                    \\"dice_vs_gt\\": scored[\\"dice_a\\"],\\n                    \\"dice_vs_baseline\\": vm.score_masks(mask, base_mask, base_mask)[\\"dice_a\\"],\\n                    \\"call\\": scored[\\"call\\"],\\n                })\\n    return rows\\n"')
_m = types.ModuleType('voxtell_mechanistic')
exec(compile(_MECH_SRC, 'voxtell_mechanistic', 'exec'), _m.__dict__)
sys.modules['voxtell_mechanistic'] = _m
_e = types.ModuleType('voxtell_experiments')
exec(compile(_EXP_SRC, 'voxtell_experiments', 'exec'), _e.__dict__)
sys.modules['voxtell_experiments'] = _e
import voxtell_mechanistic as vm
import voxtell_experiments as ve
print('experiment modules loaded')


## 2. Subjects, organs and helpers

In [ ]:
import gc

DATASET_NAME = 'neurite_oasis'
SUBJECTS = [
    ("faria", "OASIS_OAS1_0057_MR1")
]
ORGAN_CORES = ['cerebral cortex', 'cerebral white matter', 'cerebellum cortex', 'cerebellum white matter', 'lateral ventricle', 'thalamus', 'putamen', 'hippocampus', 'caudate']   # None = every bilateral organ

subject_organs = {}
for person, sid in SUBJECTS:
    rows = [r for r in load_manifest_rot000(person, DATASET_NAME) if r['subject_id'] == sid]
    rows = [r for r in rows if r['organ'].split()[0].lower() in ('left', 'right')]
    if ORGAN_CORES:
        rows = [r for r in rows if r['organ'].split(None, 1)[1] in ORGAN_CORES]
    have = {r['organ'] for r in rows}
    rows = [r for r in rows if ve.mirror(r['organ']) in have]          # keep complete left/right pairs only
    assert rows, f'no bilateral organs for {person}/{sid}'
    subject_organs[sid] = {'person': person, 'rows': rows}
print(len(subject_organs), 'subjects;', 'organs per subject:', [len(v['rows']) for v in subject_organs.values()][:3], '...')


def load_subject(sid):
    info = subject_organs[sid]
    rows = info['rows']
    base = RESEARCH_REPO / info['person'] / 'samples' / DATASET_NAME
    img_arr, _ = load_volume_ras(base / rows[0]['image_path'])
    label_vol, _ = load_seg_ras(base / rows[0]['label_path'])
    organs = [r['organ'] for r in rows]
    ids = {r['organ']: int(r['label_id']) for r in rows}
    prompts = {r['organ']: r['prompt_text'] for r in rows}
    return info['person'], img_arr, label_vol, organs, ids, prompts


def append_csv(path, rows, first):
    if not rows:
        return
    write_header = first or not path.exists()      # also right if an earlier subject produced no rows for this file
    with open(path, 'w' if first else 'a', newline='') as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        if write_header:
            w.writeheader()
        w.writerows(rows)


def tag(rows, sid, person):
    return [{'dataset': DATASET_NAME, 'person': person, 'subject_id': sid, **r} for r in rows]


## 3. Check: the custom sliding-window loop must match the predictor

In [ ]:
# The custom loop must reproduce the predictor's own masks, otherwise rows 14-16 would not be trustworthy.
_sid = next(iter(subject_organs))
_, _img, _lab, _organs, _ids, _prompts = load_subject(_sid)
_agree = ve.validate_custom_loop(predictor, _img, [_prompts[o] for o in _organs])
print(f'custom sliding-window loop vs. the predictor: {_agree * 100:.3f}% of voxels have the same sign')
assert _agree > 0.995, 'the custom loop does not match the predictor -- do not trust rows 14-16'
del _img, _lab
gc.collect(); torch.cuda.empty_cache()


## 3b. Row 0 -- baseline masks + ground truth, for plotting

Not a plan row -- a shared export run once per subject, before any experiment: the full-prompt predicted mask and the GT mask for every organ, RLE-encoded so they can be decoded and overlaid on the source image later (image + GT + prediction, side by side).

In [ ]:
import traceback

# Shared plotting export, run once per subject before any of rows 12-21: the full-prompt predicted
# mask and the GT mask for EVERY organ, RLE-encoded so you can decode and overlay them on the
# source image (image_path/label_path) slice by slice -- not just the summary metrics each row
# below computes. See scripts/voxtell_experiments.py's decode_mask_rle to reload these.
r0_path = RESULTS_ROOT / f'row0_baseline_masks_neurite_oasis.csv'
first = True
for si, sid in enumerate(subject_organs, 1):
    t0 = time.time()
    try:
        person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
        info = subject_organs[sid]
        img_path = info['rows'][0]['image_path']
        lab_path = info['rows'][0]['label_path']
        rows0 = ve.export_baseline_masks(predictor, img_arr, label_vol, organs, ids, prompts,
                                          image_path=img_path, label_path=lab_path)
        append_csv(r0_path, tag(rows0, sid, person), first)
        first = False
        print(f'[{si}/{len(subject_organs)}] {sid}: row 0 (baseline export) done in {time.time() - t0:.0f}s, {len(rows0)} organs')
    except Exception:
        print(f'[{si}/{len(subject_organs)}] {sid}: FAILED'); traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('wrote', r0_path.name)


## 4. Row 17 -- voxel-grid occlusion Shapley map

Image-region analogue of row 5's exact word-Shapley: tile a padded box around one organ's GT region into 8 blocks, noise each coalition's absent blocks, score f_dice_gt, and compute the exact Shapley credit per block.

In [ ]:
import traceback

# Row 17 is expensive (2**n_blocks model calls per organ -- default 2**8=256, same affordability
# argument as row 5's exact word-Shapley), so it runs on only the first bilateral organ per subject
# by default. Widen ROW17_ORGANS if you want more coverage. The per-block Shapley values are also
# reassembled into a voxel-shaped map (one per organ) and saved to row17_shapley_maps/<subject>.npz
# for plotting against the image/GT from row 0.
ROW17_GRID = 2          # 2**3 = 8 blocks -> 256 coalitions
ROW17_N_ORGANS = 1       # organs per subject (the first N of subject_organs[sid]['rows'])
(RESULTS_ROOT / 'row17_shapley_maps').mkdir(exist_ok=True)
r17_path = RESULTS_ROOT / f'row17_grid_shapley_neurite_oasis.csv'
first = True
for si, sid in enumerate(subject_organs, 1):
    t0 = time.time()
    try:
        person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
        rows17, maps17 = [], {}
        for organ in organs[:ROW17_N_ORGANS]:
            r, m = ve.run_grid_occlusion(predictor, img_arr, label_vol, organ, ids, prompts[organ],
                                          grid=ROW17_GRID, seed=si, return_map=True)
            rows17 += r
            maps17[organ] = m.astype(np.float16)
        append_csv(r17_path, tag(rows17, sid, person), first)
        if maps17:
            np.savez_compressed(RESULTS_ROOT / 'row17_shapley_maps' / f'{sid}.npz', **maps17)
        first = False
        print(f'[{si}/{len(subject_organs)}] {sid}: row 17 done in {time.time() - t0:.0f}s')
    except Exception:
        print(f'[{si}/{len(subject_organs)}] {sid}: FAILED'); traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('wrote', r17_path.name, 'and row17_shapley_maps/*.npz')


## 5. Row 18 -- per-layer attention/KL divergence

Reuses the same cross-attention pipeline as row 15: KL(GT mask || cross-attention density) per decoder layer, plus the same quantity after dividing out a generic prompt's attention map (isolates prompt-specific attention from wherever VoxTell looks by default).

In [ ]:
import traceback

# Attention maps (last layer and mean of layers, per organ) are saved to row18_attention_maps/<subject>.npz
# for plotting -- the CSV alone only has the scalar KL per layer, no spatial map.
(RESULTS_ROOT / 'row18_attention_maps').mkdir(exist_ok=True)
r18_path = RESULTS_ROOT / f'row18_attention_kl_neurite_oasis.csv'
first = True
for si, sid in enumerate(subject_organs, 1):
    t0 = time.time()
    try:
        person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
        rows18, maps18 = ve.run_attention_kl(predictor, img_arr, label_vol, organs, ids, prompts, return_maps=True)
        append_csv(r18_path, tag(rows18, sid, person), first)
        if maps18:
            np.savez_compressed(RESULTS_ROOT / 'row18_attention_maps' / f'{sid}.npz', **maps18)
        first = False
        print(f'[{si}/{len(subject_organs)}] {sid}: row 18 done in {time.time() - t0:.0f}s')
    except Exception:
        print(f'[{si}/{len(subject_organs)}] {sid}: FAILED'); traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('wrote', r18_path.name, 'and row18_attention_maps/*.npz')


## 6. Row 19 -- gradient-based saliency

Cheap, approximate counterpart to row 17 (one backward pass instead of 256 forward passes): vanilla-gradient saliency on the input voxels and Grad-CAM on the bottleneck feature map, for one patch centred on each organ.

In [ ]:
import traceback

# Row 19 needs real gradients -- MUST run outside inference_mode/no_grad (it is not decorated with
# either, unlike every other row in this notebook, which only ever call the model for inference).
# Saliency and Grad-CAM maps (in the crop's coordinate frame, same as row 0's masks) are saved to
# row19_saliency_maps/<subject>.npz for plotting.
(RESULTS_ROOT / 'row19_saliency_maps').mkdir(exist_ok=True)
r19_path = RESULTS_ROOT / f'row19_gradient_saliency_neurite_oasis.csv'
first = True
for si, sid in enumerate(subject_organs, 1):
    t0 = time.time()
    try:
        person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
        rows19, maps19 = [], {}
        for organ in organs:
            out = ve.run_gradient_saliency(predictor, img_arr, label_vol, organ, ids, prompts[organ], return_maps=True)
            if out is None:
                continue
            r, sal, cam = out
            rows19.append(r)
            maps19[f'{organ}__saliency'] = sal.astype(np.float16)
            maps19[f'{organ}__cam'] = cam.astype(np.float16)
        append_csv(r19_path, tag(rows19, sid, person), first)
        if maps19:
            np.savez_compressed(RESULTS_ROOT / 'row19_saliency_maps' / f'{sid}.npz', **maps19)
        first = False
        print(f'[{si}/{len(subject_organs)}] {sid}: row 19 done in {time.time() - t0:.0f}s')
    except Exception:
        print(f'[{si}/{len(subject_organs)}] {sid}: FAILED'); traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('wrote', r19_path.name, 'and row19_saliency_maps/*.npz')


## 7. Row 20 -- fused spatial feature map

Hooks VoxTellDecoder's per-stage spatial `fusion_features` and checks whether its highest-activation region moves with the prompt (matched vs. mismatched side/organ).

In [ ]:
import traceback

# Row 20's hook point (VoxTellDecoder's per-stage `fusion_features`, via seg_layers' forward_pre_hook)
# is UNVERIFIED against a live model -- the first subject's printed stage shapes below should look
# like the four documented resolutions (24, 48, 96, 192 cubed) before trusting the IoU numbers.
# The fused-feature map itself (per organ x condition) is saved to row20_fused_maps/<subject>.npz for
# plotting -- the CSV alone only has the scalar IoU, no spatial map.
(RESULTS_ROOT / 'row20_fused_maps').mkdir(exist_ok=True)
r20_path = RESULTS_ROOT / f'row20_fused_activation_overlap_neurite_oasis.csv'
first = True
_printed_shapes = False
for si, sid in enumerate(subject_organs, 1):
    t0 = time.time()
    try:
        person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
        rows20, maps20 = [], {}
        for organ in organs:
            r, m = ve.run_activation_overlap(predictor, img_arr, label_vol, organ, ids, prompts, return_maps=True)
            rows20 += r
            maps20.update(m)
        if not _printed_shapes and rows20:
            print('row 20 stages seen:', sorted({r["stage"] for r in rows20}))
            _printed_shapes = True
        append_csv(r20_path, tag(rows20, sid, person), first)
        if maps20:
            np.savez_compressed(RESULTS_ROOT / 'row20_fused_maps' / f'{sid}.npz', **maps20)
        first = False
        print(f'[{si}/{len(subject_organs)}] {sid}: row 20 done in {time.time() - t0:.0f}s')
    except Exception:
        print(f'[{si}/{len(subject_organs)}] {sid}: FAILED'); traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('wrote', r20_path.name, 'and row20_fused_maps/*.npz')


## 8. Row 21 -- joint image / organ-word / position-word Shapley

Three roles (image intact/noised, organ word, position word), 8 coalitions, one Shapley table that puts image contribution and word contribution on the same scale.

In [ ]:
import traceback

# row21_coalition_dice_neurite_oasis.csv carries the literal numbers the Shapley credit above is
# computed FROM: the actual Dice score for each of the 8 image/organ-word/position-word present
# combinations (e.g. "image only, no words at all" -> its own real Dice score) -- more intuitive to
# read directly than the derived Shapley credit, at no extra model-call cost (same 8 predictions).
ROW21_N_ORGANS = 1       # organs per subject (cheap: 8 model calls each, but keep default modest)
r21_path = RESULTS_ROOT / f'row21_joint_shapley_neurite_oasis.csv'
r21_coal_path = RESULTS_ROOT / f'row21_coalition_dice_neurite_oasis.csv'
first = True
for si, sid in enumerate(subject_organs, 1):
    t0 = time.time()
    try:
        person, img_arr, label_vol, organs, ids, prompts = load_subject(sid)
        rows21, coal21 = [], []
        for organ in organs[:ROW21_N_ORGANS]:
            r, c = ve.run_joint_image_word_shapley(predictor, img_arr, label_vol, organ, ids, seed=si, return_coalitions=True)
            rows21 += r
            coal21 += c
        append_csv(r21_path, tag(rows21, sid, person), first)
        append_csv(r21_coal_path, tag(coal21, sid, person), first)
        first = False
        print(f'[{si}/{len(subject_organs)}] {sid}: row 21 done in {time.time() - t0:.0f}s')
    except Exception:
        print(f'[{si}/{len(subject_organs)}] {sid}: FAILED'); traceback.print_exc()
    finally:
        gc.collect(); torch.cuda.empty_cache()
print('wrote', r21_path.name, 'and', r21_coal_path.name)


## 9. Summary and archive

In [ ]:
import shutil
import pandas as pd

pd.set_option('display.width', 200, 'display.max_columns', 30)
R = lambda n: pd.read_csv(RESULTS_ROOT / f'{n}_neurite_oasis.csv')

print('=== Row 0: baseline Dice per organ (sanity check before trusting any row below) ===')
d = R('row0_baseline_masks')
print(d[['organ', 'dice', 'n_pred_voxels', 'n_gt_voxels']].to_string(index=False))

print('\n=== Row 17: voxel-grid occlusion Shapley -- which block matters most, by organ ===')
d = R('row17_grid_shapley')
print(d.groupby(['organ', 'block_id']).shapley_value.mean().unstack().round(3))

print('\n=== Row 18: attention-GT KL divergence, by layer (lower = better match) ===')
d = R('row18_attention_kl')
print(d.groupby('layer')[['kl_divergence', 'kl_divergence_vs_generic_normalized']].mean().round(3))

print('\n=== Row 19: gradient saliency enrichment inside vs. outside the organ (1 = no enrichment) ===')
d = R('row19_gradient_saliency')
print(d.groupby('organ')[['saliency_enrichment', 'cam_enrichment']].mean().round(3))

print('\n=== Row 20: fused-feature activation IoU with GT, matched vs. mismatched prompt ===')
d = R('row20_fused_activation_overlap')
print(d.groupby('condition').iou_with_gt.mean().round(4))

print('\n=== Row 21: joint image / organ-word / position-word Shapley (f_dice_gt) ===')
d = R('row21_joint_shapley')
print(d[d.metric == 'f_dice_gt'].groupby('role').shapley_value.mean().round(3))

print('\n=== Row 21: raw Dice score per role-combination (the numbers the Shapley credit above comes from) ===')
dc = R('row21_coalition_dice')
print(dc[['roles_present', 'dice_gt', 'dice_self']].to_string(index=False))

zip_path = WORKING_DIR / 'voxtell_neurite_oasis_rows17_21_results'
shutil.make_archive(str(zip_path), 'zip', str(RESULTS_ROOT))
print('\nArchive:', str(zip_path) + '.zip')


## Outputs (all under `results/`)

| File | Row | Content |
|---|---|---|
| `row0_baseline_masks_neurite_oasis.csv` | - | per organ: full-prompt predicted mask and GT mask, RLE-encoded (`decode_mask_rle`), plus centroids and Dice -- the shared source for plotting image + GT + prediction overlays |
| `row17_grid_shapley_neurite_oasis.csv` | 17 | per organ and voxel-grid block: exact Shapley credit for that block's image content (the image analogue of the word-Shapley table) |
| `row17_shapley_maps/<subject>.npz` | 17 | per organ: the same Shapley credit reassembled into a full voxel-shaped map, for plotting against row 0's image/GT |
| `row18_attention_kl_neurite_oasis.csv` | 18 | per organ and layer: KL divergence between the GT mask and the cross-attention map, raw and generic-prompt-normalized |
| `row18_attention_maps/<subject>.npz` | 18 | per organ: last-layer and mean-of-layers attention density, downsampled, for plotting against row 0's image/GT |
| `row19_gradient_saliency_neurite_oasis.csv` | 19 | per organ: vanilla-gradient and Grad-CAM saliency, mean inside vs. outside the organ, and their ratio |
| `row19_saliency_maps/<subject>.npz` | 19 | per organ: the vanilla-gradient saliency map and the upsampled Grad-CAM map, voxel-shaped, for plotting |
| `row20_fused_activation_overlap_neurite_oasis.csv` | 20 | per organ and condition (matched / mismatched_side / mismatched_organ): IoU of the fused spatial feature map's hot region with the GT mask |
| `row20_fused_maps/<subject>.npz` | 20 | per organ x condition: the fused-feature map itself, upsampled to the crop frame and downsampled for storage, for plotting |
| `row21_joint_shapley_neurite_oasis.csv` | 21 | per organ, role (image / organ_word / position_word) and metric: Shapley credit on one common scale |
| `row21_coalition_dice_neurite_oasis.csv` | 21 | per organ and role-combination (`roles_present`): the literal Dice score for that combination (e.g. image only, no words at all), against ground truth (`dice_gt`) and against the full-coalition baseline (`dice_self`) -- the raw numbers the Shapley credit above is computed from |
